# 🚀 Cocopila Financial Data Agent Pipeline (Kaggle Bootstrap)

Notebook này chứa **toàn bộ mã nguồn Agent Pipeline & Thiết lập môi trường Kaggle** bao gồm:
0. **Kaggle Clone & Workspace Setup**: Clone mã nguồn từ Public Repo vào `/kaggle/working/r2AI_2026`.
1. **Môi trường & Phụ thuộc**: Cài đặt Ollama Linux Binary, Python dependencies (`qdrant-client`, `sentence-transformers`, `rank-bm25`, `langgraph`, `thefuzz`, ...) & pull model `qwen2.5-coder:1.5b`.
2. **System Configuration, Provider & Utilities**: Cấu hình hệ thống, kết nối LLM, JSON repair utility.
3. **Prompts Mẫu (YAML Prompt Templates)**: Query Parser, Code Generator & Reflection Debugging.
4. **Agent State Definition**: Shared State Dictionary dùng trong LangGraph.
5. **Toàn bộ 5 Agent Pipeline Nodes**:
   - **Node 1: Query Parser** (Phân tích câu hỏi tài chính thành JSON cấu trúc & trích xuất khái niệm cốt lõi)
   - **Node 2: Data Discovery** (Tìm kiếm bảng dữ liệu phù hợp với Search Engine & DataRegistry)
   - **Node 3: Schema Mapper** (Ánh xạ tiêu chí phụ sang tên cột thực tế trong CSV)
   - **Node 4: Code Generator & Reflection** (Sinh mã Python/Pandas trích xuất/tính toán/so sánh & tự động sửa lỗi)
   - **Node 5: AST Sandbox & Executor** (Thực thi mã Python an toàn trong Sandbox AST)
6. **Workflow StateGraph & Conditional Edge Routing**: Khởi tạo LangGraph app với vòng lặp Reflection Loop.
7. **Kiểm thử trực tiếp trên 10 câu hỏi ngẫu nhiên (seed=42)**
8. **Tổng hợp & Đóng gói Submission (Định dạng chuẩn BTC)**: Xuất file `submission.json` và đóng gói `submission.zip` chứa thư mục `data/` sẵn sàng nộp trực tiếp lên Dashboard.

## 🛠️ Section 0: Kaggle Workspace Setup & Code Cloning

In [17]:
# 0. Clone mã nguồn dự án vào thư mục /kaggle/working/r2AI_2026
import os
import sys
import shutil
import subprocess
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
REPO_DIR = WORKING_DIR / "r2AI_2026"
REPO_URL = "https://github.com/Djuybu/r2AI_2026.git"

# Kiểm tra Token nếu repo là Private (Lấy từ Kaggle Secrets hoặc biến môi trường)
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
except Exception:
    pass

if GITHUB_TOKEN and "github.com" in REPO_URL:
    auth_repo_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
else:
    auth_repo_url = REPO_URL

if os.path.exists("/kaggle/working"):
    print("🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...")
    if not REPO_DIR.exists():
        print(f"📥 Đang clone repository từ {REPO_URL} vào {REPO_DIR}...")
        res = subprocess.run(["git", "clone", auth_repo_url, str(REPO_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"⚠️ Lỗi Git Clone: {res.stderr.strip()}")
            # Dò tìm mã nguồn trong /kaggle/input làm phương án dự phòng
            dataset_candidates = list(Path("/kaggle/input").glob("**/r2AI_2026")) if Path("/kaggle/input").exists() else []
            if dataset_candidates:
                src_path = dataset_candidates[0]
                print(f"📦 Tìm thấy mã nguồn trong Kaggle Input Dataset: {src_path}. Đang sao chép sang {REPO_DIR}...")
                shutil.copytree(src_path, REPO_DIR, dirs_exist_ok=True)

    if REPO_DIR.exists():
        os.chdir(str(REPO_DIR))
        if str(REPO_DIR) not in sys.path:
            sys.path.insert(0, str(REPO_DIR))
        print(f"✅ Đã chuyển thư mục làm việc: {os.getcwd()}")
    else:
        print(f"⚠️ Không tìm thấy thư mục {REPO_DIR}. Tiếp tục với thư mục làm việc mặc định: {os.getcwd()}")
else:
    print(f"💻 Đang chạy trên môi trường Local: {os.getcwd()}")

🚀 Phát hiện môi trường Kaggle! Đang chuẩn bị thư mục làm việc...
✅ Đã chuyển thư mục làm việc: /kaggle/working/r2AI_2026


In [18]:
# Cài đặt các công cụ hệ thống hỗ trợ
import subprocess

print("📦 Cài đặt các công cụ hệ thống hỗ trợ (lshw, zstd)...")
subprocess.run(["sudo", "apt-get", "update", "-y"], check=False)
subprocess.run(["sudo", "apt", "install", "lshw", "-y"], check=False)
subprocess.run(["sudo", "apt-get", "install", "zstd", "-y"], check=False)

📦 Cài đặt các công cụ hệ thống hỗ trợ (lshw, zstd)...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
Reading package lists...

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)





Building dependency tree...
Reading state information...
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 198 not upgraded.
Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 198 not upgraded.


CompletedProcess(args=['sudo', 'apt-get', 'install', 'zstd', '-y'], returncode=0)

## 📥 Section 0.1: Installing Ollama CLI & Dependencies on Kaggle

In [19]:
# 0.1 Cài đặt Ollama CLI trên Linux kernel của Kaggle (nếu chưa có)
import subprocess

print("📥 Đang kiểm tra / cài đặt Ollama CLI trên Kaggle Linux...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=False)

📥 Đang kiểm tra / cài đặt Ollama CLI trên Kaggle Linux...


>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#######################################################################   99.7%###                                                         24.8%

######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


CompletedProcess(args='curl -fsSL https://ollama.com/install.sh | sh', returncode=0)

In [20]:
# 1. Cài đặt đầy đủ các gói phụ thuộc dự án (Bao gồm LangGraph, Qdrant Client, Sentence Transformers, BM25, ...)
import subprocess
import sys

print("📥 Đang cài đặt Python dependencies cho Agent Pipeline & RAG Search Engine...")
packages = [
    "langgraph>=0.2.0",
    "langchain-core>=0.3.0",
    "langchain-openai>=0.2.0",
    "pyyaml>=6.0",
    "json-repair>=0.30.0",
    "openpyxl>=3.1.0",
    "tabulate>=0.9.0",
    "thefuzz>=0.22.0",
    "qdrant-client",
    "sentence-transformers",
    "rank-bm25",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=False)

📥 Đang cài đặt Python dependencies cho Agent Pipeline & RAG Search Engine...


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'langgraph>=0.2.0', 'langchain-core>=0.3.0', 'langchain-openai>=0.2.0', 'pyyaml>=6.0', 'json-repair>=0.30.0', 'openpyxl>=3.1.0', 'tabulate>=0.9.0', 'thefuzz>=0.22.0', 'qdrant-client', 'sentence-transformers', 'rank-bm25'], returncode=0)

In [21]:
# 2. Khởi động Ollama Server chạy nền & Tải model
import subprocess
import time
import requests
import os

print("🚀 Đang kiểm tra/khởi động Ollama Server...")
# Kiểm tra nếu Ollama server đã chạy
server_ready = False
try:
    r = requests.get("http://localhost:11434/", timeout=2)
    if r.status_code == 200:
        server_ready = True
        print("✅ Ollama Server đã đang chạy tại port 11434!")
except Exception:
    pass

if not server_ready:
    ollama_process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    print("⏳ Chờ Ollama Server khởi động...")
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/", timeout=2)
            if r.status_code == 200:
                print("✅ Ollama Server đã sẵn sàng tại port 11434!")
                server_ready = True
                break
        except Exception:
            time.sleep(1)

MODEL_NAME = os.getenv("MODEL_NAME", "qwen3.5:9b")
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"

# Warm-up model để load trọng số vào GPU VRAM
print("🔥 Đang warm-up mô hình để tải trọng số vào bộ nhớ GPU...")
for attempt in range(3):
    try:
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": MODEL_NAME, "prompt": "Xin chao", "stream": False},
            timeout=180
        )
        if r.status_code == 200:
            print("✅ Mô hình đã sẵn sàng xử lý truy vấn!")
            break
    except Exception as e:
        print(f"ℹ️ Warm-up attempt {attempt+1}: {e}")
        time.sleep(3)


🚀 Đang kiểm tra/khởi động Ollama Server...
⏳ Chờ Ollama Server khởi động...
✅ Ollama Server đã sẵn sàng tại port 11434!
📥 Đang tải mô hình qwen3.5:9b từ Ollama registry...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dec52a44569a: 100% ▕██████████████████▏ 6.6 GB                         
pulling 7339fa418c9a: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling be595b49fe22: 100% ▕██████████████████▏  475 B                         
verifying sha256 digest 
writing manifest 
success 


✅ Đã tải thành công mô hình qwen3.5:9b!
🔥 Đang warm-up mô hình để tải trọng số vào bộ nhớ GPU...
✅ Mô hình đã sẵn sàng xử lý truy vấn!


## ⚙️ Section 1: System Configuration, LLM Provider & Utilities

In [22]:
import os
import sys
from pathlib import Path

# Ensure working directory and repo root are in sys.path
_current_dir = Path.cwd()
for _p in [_current_dir, _current_dir / "r2AI_2026", Path("/kaggle/working/r2AI_2026"), Path("/kaggle/working")]:
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import os
import sys
import io
import json
import logging
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Optional, Literal, TypedDict
from json_repair import repair_json
from langchain_openai import ChatOpenAI
from langchain_core.language_models.chat_models import BaseChatModel

def is_kaggle_environment() -> bool:
    """Check if execution environment is Kaggle."""
    return os.path.exists("/kaggle/working")

# ==============================================================================
# Dual Output Logger: Ghi toàn bộ kết quả in ra cả Console và file .txt
# ==============================================================================
class DualOutputLogger:
    """Redirects sys.stdout so that all outputs are printed to console/notebook AND saved into a .txt log file."""
    def __init__(self, log_filepath: str = "pipeline_execution.txt", stream=None):
        if stream is not None:
            self.terminal = stream
        elif isinstance(sys.stdout, DualOutputLogger):
            self.terminal = sys.stdout.terminal
        else:
            self.terminal = sys.stdout if sys.stdout is not None else sys.__stdout__
        self.log_filepath = Path(log_filepath)
        self.log_filepath.parent.mkdir(parents=True, exist_ok=True)
        self.log_file = open(self.log_filepath, "a", encoding="utf-8", errors="replace")

    def write(self, message):
        try:
            if hasattr(self.terminal, "write"):
                self.terminal.write(message)
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.write(message)
                self.log_file.flush()
        except Exception:
            pass

    def flush(self):
        try:
            if hasattr(self.terminal, "flush"):
                self.terminal.flush()
        except Exception:
            pass
        try:
            if hasattr(self, "log_file") and not self.log_file.closed:
                self.log_file.flush()
        except Exception:
            pass

    def isatty(self) -> bool:
        if hasattr(self.terminal, "isatty"):
            try:
                return self.terminal.isatty()
            except Exception:
                return False
        return False

    def fileno(self):
        if hasattr(self.terminal, "fileno"):
            return self.terminal.fileno()
        raise io.UnsupportedOperation("fileno not supported")

    def readable(self) -> bool:
        return False

    def writable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return False

    @property
    def encoding(self):
        return getattr(self.terminal, "encoding", "utf-8")

    @property
    def errors(self):
        return getattr(self.terminal, "errors", "replace")

    def close(self):
        if hasattr(self, "log_file") and not self.log_file.closed:
            self.log_file.close()

    def __getattr__(self, attr):
        return getattr(self.terminal, attr)

LOG_FILE_PATH = Path("/kaggle/working/pipeline_execution.txt") if is_kaggle_environment() else Path("./pipeline_execution.txt")
if not isinstance(sys.stdout, DualOutputLogger):
    sys.stdout = DualOutputLogger(str(LOG_FILE_PATH))

print(f"📝 Đã kích hoạt ghi log tự động ra file .txt: {LOG_FILE_PATH.resolve()}")

@dataclass
class Config:
    """System configuration parameters."""
    MODEL_NAME: str = os.getenv("MODEL_NAME", "qwen3.5:9b")
    LLM_API_BASE: str = os.getenv("LLM_API_BASE", "http://localhost:11434/v1")
    LLM_API_KEY: str = os.getenv("LLM_API_KEY", "ollama")
    TEMPERATURE: float = float(os.getenv("TEMPERATURE", "0.0"))
    MAX_TOKENS: int = int(os.getenv("MAX_TOKENS", "1024"))
    BASE_DIR: Path = Path("/kaggle/working/r2AI_2026/pipeline") if is_kaggle_environment() else Path.cwd()
    DATA_DIR: Path = Path(os.getenv("DATA_DIR", "/kaggle/working/r2AI_2026/pipeline/data"))
    PROMPTS_DIR: Path = Path(os.getenv("PROMPTS_DIR", "/kaggle/working/r2AI_2026/pipeline/src/prompts"))
    MAX_RETRIES: int = int(os.getenv("MAX_RETRIES", "3"))
    EXECUTION_TIMEOUT: int = int(os.getenv("EXECUTION_TIMEOUT", "10"))

    def get_prompt_path(self, filename: str) -> Path:
        """Get absolute path to a prompt template YAML file."""
        return self.PROMPTS_DIR / filename

config = Config()

def check_vllm_health(api_base: str, timeout: int = 3) -> bool:
    """Check if LLM server endpoint (vLLM, Ollama, or OpenAI-compatible) is responsive."""
    try:
        import requests
        base = api_base.rstrip("/")
        # 1. Check standard OpenAI /v1/models endpoint
        models_url = f"{base}/models" if base.endswith("/v1") else f"{base}/v1/models"
        try:
            r = requests.get(models_url, timeout=timeout)
            if r.status_code == 200:
                return True
        except Exception:
            pass

        # 2. Check root / or /health
        root_url = base.replace("/v1", "")
        for ep in ["", "/health"]:
            try:
                r = requests.get(f"{root_url}{ep}", timeout=timeout)
                if r.status_code == 200:
                    return True
            except Exception:
                pass
        return False
    except Exception:
        return False


def get_llm(
    cfg: Optional[Config] = None,
    temperature: Optional[float] = None,
    max_tokens: Optional[int] = None,
    timeout: Optional[int] = 120,
    **kwargs,
) -> BaseChatModel:
    """Instantiate ChatOpenAI connected to local LLM endpoint safely supporting timeout & kwargs."""
    cfg = cfg or config
    temp = temperature if temperature is not None else cfg.TEMPERATURE
    tokens = max_tokens if max_tokens is not None else cfg.MAX_TOKENS
    
    import os
    model_name = os.getenv("MODEL_NAME", cfg.MODEL_NAME)
    api_base = os.getenv("LLM_API_BASE", cfg.LLM_API_BASE)
    api_key = os.getenv("LLM_API_KEY", cfg.LLM_API_KEY)

    llm_kwargs = {
        "model": model_name,
        "base_url": api_base,
        "api_key": api_key,
        "openai_api_base": api_base,
        "openai_api_key": api_key,
        "temperature": temp,
        "max_tokens": tokens,
        "streaming": False,
        **kwargs,
    }
    if timeout is not None:
        llm_kwargs["request_timeout"] = timeout
        llm_kwargs["timeout"] = timeout
        
    return ChatOpenAI(**llm_kwargs)

def safe_parse_json(content: str) -> Dict[str, Any]:
    """Parse JSON string with automatic repair fallback."""
    if not content or not content.strip():
        return {}
    cleaned = content.strip()
    if cleaned.startswith("```json"):
        cleaned = cleaned[7:]
    elif cleaned.startswith("```"):
        cleaned = cleaned[3:]
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3]
    cleaned = cleaned.strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    try:
        repaired = repair_json(cleaned)
        return json.loads(repaired)
    except Exception as exc:
        print(f"⚠️ Safe JSON parse failed: {exc}")
        return {}

print("✅ System configuration, Logging to .txt & LLM Provider utilities loaded!")


📝 Đã kích hoạt ghi log tự động ra file .txt: /kaggle/working/pipeline_execution.txt
✅ System configuration, Logging to .txt & LLM Provider utilities loaded!


## 📝 Section 2: Prompts Mẫu (Prompt Templates)
Định nghĩa các Prompt mẫu cho Query Parser, Code Generator và Reflection Debugging Loop.

In [23]:
PROMPT_QUERY_PARSER = {
    "system_prompt": """You MUST output ONLY a valid JSON object with the exact keys: {"ticker": "string", "year": "string", "metric": "string"}.
Your output will be parsed directly by `json.loads()`. Any markdown code blocks (like ```json) or explanations WILL CRASH THE SYSTEM. Output raw JSON only.

CRITICAL INSTRUCTION: The JSON examples provided above are STRICTLY for formatting demonstration. DO NOT copy the values (ticker, year, metric) from the examples. You MUST read the actual [USER_QUERY] provided and dynamically extract the REAL ticker, REAL year, and REAL metric requested by the user.

CRITICAL NEGATIVE RULES:
- NEVER output corporate words like 'CTCP', 'TMCP', 'TẬP ĐOÀN', 'CÔNG TY', 'NGÂN HÀNG', 'TỔNG CÔNG TY', 'TNHH', 'JSC' as the ticker!
- Ticker is ALWAYS a specific 3-5 uppercase letter stock code (e.g. NVL, HHV, DLG, BVH, VJC, VCB, MBB, EIB, FPT, FTS).
- If the company name is 'CTCP Tập đoàn Đầu tư Địa ốc No Va', the ticker is 'NVL', NOT 'CTCP'.
- If the company name is 'CTCP Chứng khoán FPT', the ticker is 'FTS', NOT 'FPT' and NOT 'CTCP'.
- If the company name is 'CTCP Đầu tư Hạ tầng Giao thông Đèo Cả', the ticker is 'HHV', NOT 'CTCP'.
- If the company name is 'CTCP Tập đoàn Đức Long Gia Lai', the ticker is 'DLG', NOT 'CTCP'.

## DETAILED INSTRUCTIONS:
1. "ticker": Map the target company or bank name in the query to its exact 3-5 letter uppercase ticker symbol using the stock mapping (code_stock.csv). (e.g., "Vietjet" -> "VJC", "Ngân hàng TMCP Sài Gòn Thương Tín" -> "STB", "FPT" -> "FPT", "Tập đoàn Vingroup" -> "VIC", "Novaland" / "Địa ốc No Va" -> "NVL", "Đèo Cả" -> "HHV", "Đức Long Gia Lai" -> "DLG", "Eximbank" / "EIB" -> "EIB"). If no company is mentioned or if the company is unknown, output an empty string "". DO NOT output null or None!

2. "year": Extract the target year or list of years from the query as a string (e.g., "2023" or "2021, 2022, 2023"). If no year is found, output an empty string "". DO NOT output null or None!

3. "metric": Extract the exact financial metric string verbatim from the query (e.g., "Lãi tiền gửi", "Chi phí khác", "Doanh thu thuần", "Lãi vay phải trả", "Tổng quỹ lương", "Phải thu ngắn hạn khác", "Giá gốc chứng khoán kinh doanh", "Tổng tỷ lệ quyền biểu quyết"). Do NOT copy metrics from examples!

OUTPUT FORMAT: Output ONLY raw valid JSON matching {"ticker": "string", "year": "string", "metric": "string"}.""",
    "user_prompt_template": """Câu hỏi: {user_query}

Ví dụ mẫu:
{few_shot_examples}

Hãy phân tích câu hỏi trên và trả về JSON:""",
    "few_shot_examples": [
        {
            "user_query": "Lãi tiền gửi năm 2021 của Vietjet (VJC) là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "VJC", "year": "2021", "metric": "Lãi tiền gửi"}'
        },
        {
            "user_query": "Chi phí lương và các khoản khác theo lương của công ty mẹ CTCP Chứng khoán FPT trong năm 2021 là bao nhiêu tỷ đồng?",
            "parsed_output": '{"ticker": "FTS", "year": "2021", "metric": "Chi phí lương và các khoản khác theo lương"}'
        },
        {
            "user_query": "Lãi vay phải trả của CTCP Tập đoàn Đức Long Gia Lai (DLG) cuối năm 2023 là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "DLG", "year": "2023", "metric": "Lãi vay phải trả"}'
        },
        {
            "user_query": "Doanh thu hoạt động tài chính năm 2020 của Ngân hàng TMCP Sài Gòn Thương Tín (STB)",
            "parsed_output": '{"ticker": "STB", "year": "2020", "metric": "Doanh thu hoạt động tài chính"}'
        },
        {
            "user_query": "Vốn chủ sở hữu của FIT là bao nhiêu tỷ đồng vào ngày 31/12/2015?",
            "parsed_output": '{"ticker": "FIT", "year": "2015", "metric": "Vốn chủ sở hữu"}'
        },
        {
            "user_query": "Tổng quỹ lương năm 2022 của công ty mẹ EIB là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "EIB", "year": "2022", "metric": "Tổng quỹ lương"}'
        },
        {
            "user_query": "Tổng tỷ lệ quyền biểu quyết của công ty mẹ CTCP Đầu tư Hạ tầng Giao thông Đèo Cả năm 2023 là bao nhiêu phần trăm?",
            "parsed_output": '{"ticker": "HHV", "year": "2023", "metric": "Tổng tỷ lệ quyền biểu quyết"}'
        },
        {
            "user_query": "Tổng phải thu ngắn hạn khác của công ty mẹ CTCP Tập đoàn Đầu tư Địa ốc No Va đến ngày 31 tháng 12 năm 2016 là bao nhiêu triệu đồng?",
            "parsed_output": '{"ticker": "NVL", "year": "2016", "metric": "Phải thu ngắn hạn khác"}'
        }
    ]
}

PROMPT_SCHEMA_MAPPER = {
    "system_prompt": """Bạn là chuyên gia phân tích dữ liệu tài chính Việt Nam.
Nhiệm vụ của bạn là tư duy, phân tích cấu trúc bảng và ý nghĩa ngữ nghĩa của từng cột số liệu trong bảng báo cáo tài chính.

QUY TRÌNH TƯ DUY KHI TẠO MÔ TẢ CỘT:
1. Xác định nội dung & chủ đề của bảng (bỏ qua số thứ tự mục, ví dụ: "29. Doanh thu hoạt động tài chính" -> "Doanh thu hoạt động tài chính").
2. Phân tích chiều thời gian / kỳ kế toán của cột (năm cụ thể như 2020, đầu năm, cuối năm, ngày kết thúc...).
3. ĐẶC BIỆT NHẬN DIỆN ĐƠN VỊ ĐO LƯỜNG trong tên cột hoặc bảng:
   - Nếu tên cột có từ "Triệu", "triệu" (như "Năm 2020Triệu", "Số cuối nămTriệu", "2021Triệu") -> Bạn PHẢI giải thích rõ đơn vị tính là triệu đồng / triệu VND để model ở các bước sau hiểu giá trị thực tế của số đo.
   - Nếu tên cột có từ "Tỷ", "Nghìn", "USD", "VND" -> Bạn PHẢI chỉ rõ đơn vị tính tương ứng (ví dụ: đơn vị đo: tỷ đồng, đơn vị đo: USD).
4. Tạo mô tả ngắn gọn (1 câu tiếng Việt) kết hợp TÊN BẢNG + THỜI KỲ + ĐƠN VỊ TÍNH (nếu có).

Ví dụ:
- Bảng: "Doanh thu hoạt động tài chính", Cột: "2018" -> "Doanh thu hoạt động tài chính của các nội dung trong năm 2018"
- Bảng: "NGHĨA VỤ NỢ TIỀM ẨN", Cột: "Số cuối nămTriệu" -> "Nghĩa vụ nợ tiềm ẩn của các nội dung vào thời điểm cuối năm (đơn vị đo: triệu đồng)"
- Bảng: "Thu nhập lãi thuần", Cột: "Năm 2020Triệu" -> "Thu nhập lãi thuần của các nội dung trong năm 2020 (đơn vị đo: triệu đồng)"
- Bảng: "Bảng cân đối kế toán", Cột: "31/12/2022" -> "Số liệu cân đối kế toán của các chỉ tiêu tại ngày 31/12/2022"

Trả về ONLY một JSON array với format:
[{"column_name": "tên cột", "column_description": "mô tả chi tiết kết hợp tên bảng, thời kỳ và đơn vị đo"}]
KHÔNG giải thích ngoài lề, KHÔNG bọc markdown. Output raw JSON only.""",
    "user_prompt_template": """Bảng báo cáo tài chính: {table_name}
Đơn vị tính từ metadata (nếu có): {don_vi_tinh}
Danh sách cột số liệu cần phân tích:
{columns_info}
Hãy phân tích và đưa ra mô tả chi tiết cho từng cột (kết hợp tên bảng, kỳ báo cáo và làm rõ đơn vị đo lường nếu có):"""
}

PROMPT_CODE_GENERATOR = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis chuyên sâu về Báo cáo Tài chính Việt Nam.
Nhiệm vụ của bạn là sinh ra đoạn mã Python để truy vấn dữ liệu từ bảng báo cáo tài chính.

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without `.astype(str)` or without explicitly passing `regex=False`.
- You are FORBIDDEN from redefining `clean_val` or `extract_value`. They are pre-injected into your execution scope.
</BANNED_SYNTAX>

CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table"). You MUST let the script crash if the exact metric is not found!
CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.

QUY TẮC BẮT BUỘC (TUYỆT ĐỐI TUÂN THỦ):
1. Dữ liệu nằm trong file CSV. Đọc file bằng `pd.read_csv(file_path)`.
2. Cấu trúc bảng: Cột nhãn (label_column) chứa tên chỉ tiêu. Cột giá trị (value_column) chứa số liệu.
3. BẮT BUỘC dùng `regex=False` khi sử dụng `str.contains` để tránh lỗi regex với các chỉ tiêu có dấu ngoặc tròn.
4. BẮT BUỘC áp dụng MẪU TRUY VẤN ĐA CẤP ĐỘ (MULTI-STAGE SAFE PANDAS QUERY PATTERN):
   - Cấp 1 (Exact match): Thử khớp toàn bộ chuỗi chỉ tiêu làm sạch.
   - Cấp 2 (Core keyword match): Nếu Cấp 1 rỗng, thử từ khóa cốt lõi rút gọn (bỏ các tiền tố "Tổng", "Tổng số", "Số dư", "Chi phí").
   - Cấp 3 (Total/Summary fallback): Nếu vẫn rỗng và câu hỏi tìm số dư/tổng thể, thử dòng 'TỔNG CỘNG' / 'CỘNG'.
   ```python
   # Cấp 1: Thử khớp chính xác
   filtered_df = df[df['{label_col}'].astype(str).str.contains('{noi_dung}', case=False, na=False, regex=False)]
   # Cấp 2: Nếu rỗng, thử từ khóa cốt lõi
   if filtered_df.empty and len(core_tokens) > 0:
       filtered_df = df[df['{label_col}'].astype(str).str.contains(core_tokens[0], case=False, na=False, regex=False)]
   # Cấp 3: Nếu rỗng, thử dòng tổng cộng (đối với câu hỏi tổng/toàn bộ)
   if filtered_df.empty:
       filtered_df = df[df['{label_col}'].astype(str).str.contains('tổng cộng|tổng số|cộng', case=False, na=False, regex=True)]

   if not filtered_df.empty:
       result = extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])
   else:
       raise ValueError("Metric not found in table")
   ```
5. QUY TẮC ENTITY-AWARE FILTER (NHÂN SỰ / THÙ LAO / BAN LÃNH ĐẠO):
   - Khi câu hỏi yêu cầu thù lao, tiền lương, hoặc quyền biểu quyết của cá nhân (ví dụ: "Chu Thị Bình", "Trương Gia Bình"):
   - Sinh mã lọc trực tiếp theo tên cá nhân trên cột nhãn:
     ```python
     filtered_df = df[df['{label_col}'].astype(str).str.contains('Chu Thị Bình', case=False, na=False, regex=False)]
     if not filtered_df.empty:
         result = extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])
     else:
         raise ValueError("Metric not found in table")
     ```
6. QUY TẮC SỬ DỤNG SCHEMA:
   - Nếu bảng có NHIỀU HƠN MỘT CỘT GIÁ TRỊ, so sánh chỉ tiêu cần tìm với tên và mô tả của từng cột trong SCHEMA để chọn đúng cột giá trị phù hợp nhất.
   - Nếu dữ liệu cần tìm là một SECTION (danh mục) trong SCHEMA:
     + Ưu tiên lấy trực tiếp `total_value` của section đó nếu có.
     + Nếu `total_value` không có hoặc rỗng, thực hiện tính tổng các hàng nằm trong `range` [start, end] của section đó trên cột giá trị: `df.iloc[start:end+1]`.
7. XỬ LÝ KHI KHÔNG TÌM THẤY DỮ LIỆU / DỮ LIỆU RỖNG:
   - Khi không lọc thấy dòng hoặc extract_value không tìm thấy giá trị, BẮT BUỘC raise `ValueError("Metric not found in table")` để kích hoạt Reflection Loop.
8. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.
9. CHỈ trả về code Python thuần túy. KHÔNG giải thích, KHÔNG bọc markdown.""",
    "goal_descriptions": {
        "trich_xuat": "TRÍCH XUẤT giá trị cụ thể",
        "tinh_tong": "TÍNH TỔNG (tìm dòng Tổng/Cộng trước, hoặc tính tổng section theo range)",
        "so_sanh": "SO SÁNH giá trị giữa nhiều năm/công ty"
    },
    "goal_instructions": {
        "trich_xuat": """HƯỚNG DẪN CỤ THỂ (TRÍCH XUẤT):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Nếu nội dung cần tìm là một section: kiểm tra nếu có total_value thì lấy total_value, nếu không thì cộng các dòng trong range [start:end+1].
3. Áp dụng Multi-Stage Safe Query Pattern trên cột '{label_col}' (Cấp 1: Khớp chính xác '{noi_dung}' -> Cấp 2: Khớp từ khóa cốt lõi -> Cấp 3: Khớp 'tổng cộng').
4. Dùng kiểm tra `if not filtered_df.empty:` để lấy giá trị qua `extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])`. Nếu rỗng → raise ValueError("Metric not found in table").
5. Gán vào biến result.""",
        "tinh_tong": """HƯỚNG DẪN CỤ THỂ (TÍNH TỔNG):
1. Đọc file CSV bằng pd.read_csv(file_path).
2. Nếu nội dung cần tìm là một section: kiểm tra nếu có total_value thì lấy total_value, nếu không thì lấy `sub_df = df.iloc[start:end+1]` và tính tổng các dòng qua `clean_val`.
3. Ngược lại, tìm dòng có chứa 'Tổng' hoặc 'Cộng' hoặc tên chỉ tiêu '{noi_dung}' ở cột '{label_col}' bằng Multi-Stage Safe Query Pattern.
4. Kiểm tra `if not filtered_df.empty:` để lấy giá trị qua `extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])`. Nếu không tìm thấy, cộng các dòng con liên quan. Nếu vẫn rỗng → raise ValueError("Metric not found in table").
5. Gán kết quả vào result.""",
        "so_sanh": """HƯỚNG DẪN CỤ THỂ (SO SÁNH NĂM / TÍNH TỐC ĐỘ TĂNG TRƯỞNG):
1. Đọc từng file CSV cho từng năm (ví dụ file_path_2019, file_path_2020, file_path_2021...).
2. Truy vấn hàng ở cột '{label_col}' với `df['{label_col}'].astype(str).str.contains(..., case=False, na=False, regex=False)` và lấy giá trị từng năm qua `extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])`. Nếu rỗng → raise ValueError("Metric not found in table").
3. Tính tốc độ tăng trưởng phần trăm (%) giữa năm đầu và năm cuối: `growth_rate = ((val_last - val_first) / val_first) * 100`.
4. Gán kết quả vào `result`."""
    },
    "user_prompt_template": """Yêu cầu người dùng: {user_query}

MỤC TIÊU: {muc_tieu_desc}
NỘI DUNG cần tìm (ở cột label): '{noi_dung}'
Công ty: {ten_cong_ty}
Năm: {so_nam}
Tiêu chí phụ: {tieu_chi_phu}

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn (chứa tên chỉ tiêu): '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{goal_instruction}

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without `.astype(str)` or without explicitly passing `regex=False`.
</BANNED_SYNTAX>

🚨 BẮT BUỘC KHÔNG ĐƯỢC VI PHẠM:
1. CRITICAL RULE: DO NOT EVER hardcode or append result = 0.0 or any fallback value at the end of the script to bypass errors. If the filtered dataframe is empty, the script MUST end by raising a ValueError("Metric not found in table").
2. CRITICAL OUTPUT RULE: Output ONLY executable Python code. NO markdown formatting. NO ```python tags. NO conversational text or explanations. Start directly with 'import pandas'.
3. Đọc file bằng pd.read_csv(file_path...).
4. Dùng mẫu Multi-Stage Safe Pattern với `regex=False` và `extract_value(..., _df=df, _row_idx=...)`.
5. BẮT BUỘC dùng `if not filtered_df.empty:` trước khi lấy `.iloc[0]`. Nếu rỗng, BẮT BUỘC `raise ValueError("Metric not found in table")`. KHÔNG ĐƯỢC gán result = 0.0!
6. KHÔNG ĐƯỢC filter theo `df['Ma_Doanh_Nghiep'] == ...` vì dữ liệu đã đúng công ty.
7. CHỈ ĐƯỢC SỬ DỤNG CÁC BIẾN ĐƯỜNG DẪN FILE ĐÃ ĐƯỢC ĐỊNH NGHĨA Ở TRÊN:
{paths_str}
8. Kết quả cuối cùng BẮT BUỘC lưu vào biến `result`.""",
    "few_shot_examples": [
        {
            "user_query": "Doanh thu thuần năm 2023 của FPT",
            "file_path": "data/FPT_2023.csv",
            "column_mapping": '{"label_column": "CHỈ TIÊU", "value_column": "Năm nay"}',
            "generated_code": """import pandas as pd

df = pd.read_csv(file_path)
filtered_df = df[df['CHỈ TIÊU'].astype(str).str.contains('Doanh thu thuần', case=False, na=False, regex=False)]
if filtered_df.empty:
    filtered_df = df[df['CHỈ TIÊU'].astype(str).str.contains('Doanh thu', case=False, na=False, regex=False)]
if not filtered_df.empty:
    result = extract_value(filtered_df.iloc[0], 'Năm nay', _df=df, _row_idx=filtered_df.index[0])
else:
    raise ValueError("Metric not found in table")"""
        },
        {
            "user_query": "Thù lao của Chu Thị Bình trong năm 2021 tại MPC",
            "file_path": "data/MPC_2021.csv",
            "column_mapping": '{"label_column": "Họ và tên", "value_column": "Thù lao"}',
            "generated_code": """import pandas as pd

df = pd.read_csv(file_path)
filtered_df = df[df['Họ và tên'].astype(str).str.contains('Chu Thị Bình', case=False, na=False, regex=False)]
if not filtered_df.empty:
    result = extract_value(filtered_df.iloc[0], 'Thù lao', _df=df, _row_idx=filtered_df.index[0])
else:
    raise ValueError("Metric not found in table")"""
        }
    ]
}

PROMPT_REFLECTION = {
    "system_prompt": """Bạn là một chuyên gia lập trình Python & Pandas Data Analysis.
Nhiệm vụ của bạn là SỬA LỖI mã nguồn Python đã sinh ra ở bước trước dựa trên thông báo lỗi (Traceback) và mẫu dữ liệu thực tế từ bảng báo cáo tài chính.

<BANNED_SYNTAX>
- You are FORBIDDEN from writing `result = 0.0` or `result = 0` or any fallback value at the end of the script to bypass errors.
- You are FORBIDDEN from using `.iloc[0]` or `.values[0]` outside of an `if not filtered_df.empty:` block.
- You are FORBIDDEN from using `str.contains()` without `.astype(str)` or without explicitly passing `regex=False`.
</BANNED_SYNTAX>

🚨 NGUYÊN NHÂN LỖI THƯỜNG GẶP & CÁCH SỬA:
1. `ValueError: Metric not found in table`:
   - Chuỗi tìm kiếm trong `str.contains` quá dài hoặc không khớp chính xác với chỉ tiêu trong bảng.
   - HÃY NHÌN VÀO DANH SÁCH MẪU CHỈ TIÊU THỰC TẾ (bên dưới) để chọn từ khóa ngắn gọn, cốt lõi hơn hoặc áp dụng Multi-Stage Safe Query Pattern.
2. `re.error: missing ), unterminated subpattern`:
   - BẮT BUỘC thêm `regex=False` vào `str.contains(..., regex=False)`.

QUY TẮC BẮT BUỘC:
1. BẮT BUỘC dùng `regex=False` trong mọi lệnh `str.contains`.
2. BẮT BUỘC kiểm tra `if not filtered_df.empty:` trước khi truy xuất giá trị qua `extract_value(filtered_df.iloc[0], '{value_col}', _df=df, _row_idx=filtered_df.index[0])`. Nếu rỗng, BẮT BUỘC `raise ValueError("Metric not found in table")`. KHÔNG ĐƯỢC gán `result = 0.0`!
3. CHỈ trả về code Python thuần túy. KHÔNG giải thích, KHÔNG markdown.""",
    "user_prompt_template": """Yêu cầu người dùng: {user_query}
Mục tiêu: {muc_tieu}
Nội dung cần tìm: '{noi_dung}'

DỮ LIỆU CÓ SẴN:
{files_context}

CỘT QUAN TRỌNG:
- Cột nhãn: '{label_col}'
- Cột giá trị: '{value_col}'

BIẾN ĐƯỜNG DẪN FILE:
{paths_str}

{sample_labels_str}

MÃ NGUỒN CŨ BỊ LỖI:
```python
{previous_code}
```

THÔNG BÁO LỖI / TRACEBACK:
{error_traceback}

Hãy phân tích lỗi, nhìn vào mẫu chỉ tiêu thực tế, và viết lại toàn bộ mã Python sửa lỗi:"""
}

print("✅ Prompt templates loaded (Query Parser, Schema Mapper, Code Generator, Reflection)!")

✅ Prompt templates loaded (Query Parser, Schema Mapper, Code Generator, Reflection)!


## 🧠 Section 3: Agent State Definition

In [24]:
"""State definition for LangGraph workflow in Cocopila."""

from typing import Any, Dict, List, Literal, Optional, TypedDict


class AgentState(TypedDict, total=False):
    """Shared state dictionary passed across LangGraph nodes."""

    # User Input
    user_query: str

    # Node 1: Query Parser Output (structured)
    parsed_query: Dict[str, Any]
    # Expected format:
    # {
    #   "muc_tieu": "trich_xuat" | "tinh_tong" | "so_sanh",
    #   "noi_dung": str,          # Nội dung ở cột đầu tiên của bảng
    #   "ten_cong_ty": str,       # Tên công ty
    #   "so_nam": list[str],      # Danh sách năm
    #   "tieu_chi_phu": str | None  # Tiêu chí phụ (tên cột giá trị)
    # }

    # Node 2: Data Discovery Output
    discovered_tables: List[Dict[str, Any]]  # List bảng từ Search Engine
    matched_table_path: Optional[str]        # Đường dẫn bảng Top-1 (hoặc bảng tốt nhất)
    table_schema: List[str]                  # Danh sách tên cột của bảng tốt nhất
    first_row_values: Dict[str, str]         # Giá trị hàng đầu tiên (khi cột có tên là số)

    # Node 3: Schema Mapper Output
    column_mapping: Dict[str, str]  # Map tiêu_chí_phụ → tên cột thực tế
    schema: Dict[str, Any]          # Schema phân tích bảng: useful_columns + sub_sections

    # Node 4: Code Generator Output
    generated_code: str

    # Node 5: Executor Output
    execution_result: Any
    error_traceback: Optional[str]
    retry_count: int

    # Multi-Table Top-5 Candidate Execution & Aggregator
    multi_table_results: List[Dict[str, Any]]
    top_k_candidates: List[Dict[str, Any]]
    current_table_index: int
    aggregated_value: Optional[Any]

    # General Workflow Metadata
    status: Literal["pending", "success", "error"]
    error_message: Optional[str]
    node_latencies: Dict[str, float]


print('✅ Section 3: Agent State loaded!')


✅ Section 3: Agent State loaded!


## 🧩 Section 4: Agent Pipeline Nodes

### Node 1: Query Parser Node
Trích xuất `ten_cong_ty`, `so_nam`, `noi_dung`, `thao_tac`, `tieu_chi_phu` từ câu hỏi người dùng.

In [25]:
"""Node 1: Query Parser Node.
Phân tích câu hỏi tài chính thành cấu trúc JSON chuẩn.
Output format: ten_cong_ty, so_nam, noi_dung, thao_tac (muc_tieu: trich_xuat | so_sanh), tieu_chi_phu.
Không thực hiện tìm bảng — việc này do Data Discovery xử lý.
"""

import re
import time
import yaml
from pathlib import Path
from typing import Dict, Any, Optional, List, Tuple, Set
import pandas as pd
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config
from pipeline.src.llm_provider import get_llm
from pipeline.src.utils.json_repair import safe_parse_json


def load_query_parser_prompt(cfg: Config) -> Dict[str, Any]:
    """Load prompt templates and few-shot examples from YAML with fallback to Kaggle globals."""
    try:
        if hasattr(cfg, "get_prompt_path"):
            prompt_path = cfg.get_prompt_path("query_parser.yaml")
            if prompt_path and Path(prompt_path).exists():
                with open(prompt_path, "r", encoding="utf-8") as f:
                    return yaml.safe_load(f)
    except Exception:
        pass

    if "PROMPT_QUERY_PARSER" in globals():
        return globals()["PROMPT_QUERY_PARSER"]

    raise FileNotFoundError("Prompt file query_parser.yaml not found and no global fallback available.")


NEGATIVE_BLOCKLIST: Set[str] = {
    # Corporate prefixes / legal forms
    "CTCP", "TMCP", "TẬP ĐOÀN", "CÔNG TY", "NGÂN HÀNG", "TỔNG CÔNG TY",
    "TNHH", "CP", "DN", "JSC", "CORP", "CORPORATION", "GROUP", "BANK",
    "HOLDINGS", "SECURITIES", "CHỨNG KHOÁN", "BẢO HIỂM", "BẤT ĐỘNG SẢN",
    # Financial report terms & abbreviations
    "BÁO CÁO", "TÀI CHÍNH", "BCTC", "TCTD", "TNDN", "TNCN", "GTGT", "VAMC",
    "CHKQT", "CHK", "HĐQT", "HDQT", "TSCĐ", "TSCD", "SXKD", "XDCB",
    # Generic domain words
    "VIỆT NAM", "QUỐC TẾ", "ĐẦU TƯ", "THƯƠNG MẠI", "XÂY DỰNG", "NĂNG LƯỢNG",
    # Currencies & exchanges
    "VND", "USD", "EUR", "HOSE", "HNX", "UPCOM", "VN30", "VNINDEX",
}

ALIAS_TICKER_MAP: Dict[str, str] = {
    # Real Estate & Construction
    "địa ốc no va": "NVL",
    "đầu tư địa ốc no va": "NVL",
    "tập đoàn đầu tư địa ốc no va": "NVL",
    "novaland": "NVL",
    "nova": "NVL",
    "hạ tầng giao thông đèo cả": "HHV",
    "giao thông đèo cả": "HHV",
    "đèo cả": "HHV",
    "đức long gia lai": "DLG",
    "vincom retail": "VRE",
    "vingroup": "VIC",
    "đất xanh": "DXG",
    "bất động sản đất xanh": "DXS",
    "nam long": "NLG",
    "phát đạt": "PDR",
    "kinh bắc": "KBC",
    "hà đô": "HDG",
    "hòa bình": "HBC",
    "xây dựng hòa bình": "HBC",
    "sunshine homes": "SSH",
    "sunshine": "SSH",
    "bất động sản thế kỷ": "CRE",
    "cenland": "CRE",
    "bất động sản văn phú": "VPI",
    "văn phú invest": "VPI",
    "khải hoàn land": "KHG",
    "hải phát": "HPX",
    "tasco": "HUT",
    "sông đà": "SJG",
    "sonadezi": "SNZ",
    "becamex ijc": "IJC",

    # Banking & Finance
    "an bình": "ABB",
    "ab bank": "ABB",
    "abbank": "ABB",
    "á châu": "ACB",
    "bắc á": "BAB",
    "baca bank": "BAB",
    "đầu tư và phát triển việt nam": "BID",
    "bidv": "BID",
    "bảo việt": "BVH",
    "tập đoàn bảo việt": "BVH",
    "công thương việt nam": "CTG",
    "vietinbank": "CTG",
    "xuất nhập khẩu việt nam": "EIB",
    "eximbank": "EIB",
    "phát triển thành phố hồ chí minh": "HDB",
    "hdbank": "HDB",
    "kiên long": "KLB",
    "kienlongbank": "KLB",
    "quân đội": "MBB",
    "mbbank": "MBB",
    "hàng hải việt nam": "MSB",
    "nam á": "NAB",
    "nama bank": "NAB",
    "quốc dân": "NVB",
    "ncb": "NVB",
    "phương đông": "OCB",
    "sài gòn công thương": "SGB",
    "saigonbank": "SGB",
    "sài gòn - hà nội": "SHB",
    "shb": "SHB",
    "đông nam á": "SSB",
    "seabank": "SSB",
    "sài gòn thương tín": "STB",
    "sacombank": "STB",
    "sài gòn tài lộc": "STB",
    "việt á": "VAB",
    "viet a bank": "VAB",
    "ngoại thương việt nam": "VCB",
    "vietcombank": "VCB",
    "quốc tế việt nam": "VIB",
    "việt nam thịnh vượng": "VPB",
    "vpbank": "VPB",
    "evn finance": "EVF",

    # Securities
    "chứng khoán fpt": "FTS",
    "chứng khoán mb": "MBS",
    "chứng khoán ssi": "SSI",

    # Technology & Telecommunications
    "fpt": "FPT",
    "viễn thông fpt": "FOX",
    "fpt telecom": "FOX",

    # Retail & Consumer Goods
    "thế giới di động": "MWG",
    "sữa việt nam": "VNM",
    "vinamilk": "VNM",
    "sabeco": "SAB",
    "bia sài gòn": "SAB",
    "masan": "MSN",
    "tập đoàn masan": "MSN",
    "hàng tiêu dùng masan": "MCH",
    "masan consumer": "MCH",
    "masan meatlife": "MML",
    "masan high-tech materials": "MSR",
    "vàng bạc đá quý phú nhuận": "PNJ",
    "pnj": "PNJ",
    "đường quảng ngãi": "QNS",
    "vinasoy": "QNS",
    "dabaco": "DBC",

    # Energy, Industry & Materials
    "hòa phát": "HPG",
    "hoa sen": "HSG",
    "nam kim": "NKG",
    "xăng dầu việt nam": "PLX",
    "petrolimex": "PLX",
    "lọc hóa dầu việt nam": "BSR",
    "lọc dầu bình sơn": "BSR",
    "bình sơn": "BSR",
    "khí việt nam": "GAS",
    "pv gas": "GAS",
    "điện lực dầu khí": "POW",
    "pv power": "POW",
    "vận tải dầu khí": "PVT",
    "pvtrans": "PVT",
    "phân bón dầu khí cà mau": "DCM",
    "đạm cà mau": "DCM",
    "phân bón và hóa chất dầu khí": "DPM",
    "đạm phú mỹ": "DPM",
    "cao su việt nam": "GVR",
    "công nghiệp cao su việt nam": "GVR",
    "cảng hàng không việt nam": "ACV",
    "cảng hàng không quốc tế": "ACV",
    "hàng không vietjet": "VJC",
    "vietjet": "VJC",
    "vietjet air": "VJC",
    "dệt may việt nam": "VGT",
    "vinatex": "VGT",
    "viglacera": "VGC",
    "gelex": "GEX",
    "tập đoàn gelex": "GEX",
    "điện lực gelex": "GEE",
    "gelex electric": "GEE",
    "điện gia lai": "GEG",
    "nhiệt điện hải phòng": "HND",
    "điện lực tkv": "DTK",
    "thủy điện đa nhim": "DNH",
    "tập đoàn pc1": "PC1",
    "xây lắp điện 1": "PC1",
    "vicem hà tiên": "HT1",
    "xi măng vicem hà tiên": "HT1",
    "xi măng hà tiên": "HT1",
    "hà tiên 1": "HT1",
    "nhựa an phát xanh": "AAA",
    "an phát xanh": "AAA",
    "an phát": "AAA",
    "thủy sản minh phú": "MPC",
    "minh phú": "MPC",
    "sao mai": "ASM",
    "tập đoàn sao mai": "ASM",
    "hoàng anh gia lai": "HAG",
    "hagl": "HAG",
    "nông nghiệp quốc tế hoàng anh gia lai": "HNG",
    "hagl agrico": "HNG",
    "nông nghiệp baf": "BAF",
    "baf việt nam": "BAF",
    "container việt nam": "VSC",
    "viconship": "VSC",
    "lương thực miền nam": "VSF",
    "vinafood 2": "VSF",
    "vinafood ii": "VSF",
    "lâm nghiệp việt nam": "VIF",
    "vinafor": "VIF",
    "đại dương": "OGC",
    "tập đoàn đại dương": "OGC",
    "sam holdings": "SAM",
    "gỗ trường thành": "TTF",
}

_CACHED_NAME_TO_CODE: Optional[List[Tuple[str, str]]] = None
_CACHED_ALL_TICKERS: Optional[Set[str]] = None


def _get_stock_mappings() -> Tuple[List[Tuple[str, str]], Set[str]]:
    """Load and cache (name_to_code, all_tickers) from code_stock.csv."""
    global _CACHED_NAME_TO_CODE, _CACHED_ALL_TICKERS
    if _CACHED_NAME_TO_CODE is not None and _CACHED_ALL_TICKERS is not None:
        return _CACHED_NAME_TO_CODE, _CACHED_ALL_TICKERS

    name_to_code: List[Tuple[str, str]] = []
    all_tickers: Set[str] = set()

    possible_paths = [
        Path("r2AI_2026/rag_module/code_stock.csv"),
        Path("r2AI_2026/rag_module/ViFinQA/code_stock.csv"),
        Path("rag_module/code_stock.csv"),
        Path("rag_module/ViFinQA/code_stock.csv"),
        Path("/kaggle/working/r2AI_2026/rag_module/code_stock.csv"),
        Path(__file__).resolve().parent.parent.parent.parent / "rag_module" / "code_stock.csv",
        Path(__file__).resolve().parent.parent.parent.parent / "rag_module" / "ViFinQA" / "code_stock.csv",
        Path(__file__).resolve().parent.parent.parent / "rag_module" / "code_stock.csv",
        Path(__file__).resolve().parent.parent.parent / "rag_module" / "ViFinQA" / "code_stock.csv",
    ]

    csv_path = None
    for p in possible_paths:
        if p.exists():
            csv_path = p
            break

    if csv_path:
        try:
            df = pd.read_csv(csv_path, encoding="utf-8-sig", dtype=str)
            ticker_col = next((c for c in df.columns if "CK" in c.upper()), df.columns[0])
            name_col = next((c for c in df.columns if "TÊN" in c.upper() or "TEN" in c.upper()), df.columns[1])

            for _, row in df.iterrows():
                code = str(row[ticker_col]).strip().upper() if pd.notna(row[ticker_col]) else ""
                name = str(row[name_col]).strip() if pd.notna(row[name_col]) else ""
                if code:
                    all_tickers.add(code)
                if code and name:
                    name_to_code.append((name, code))
                    clean_name = re.sub(
                        r"\b(CTCP|Tập đoàn|Công ty|Cổ phần|Ngân hàng|TMCP|\-\s*CTCP|Tổng Công ty|TNHH)\b",
                        "",
                        name,
                        flags=re.IGNORECASE
                    ).strip(" -")
                    if clean_name and clean_name.lower() != name.lower() and len(clean_name) >= 3:
                        name_to_code.append((clean_name, code))
        except Exception as e:
            print(f"⚠️ [Query Parser] Error loading code_stock.csv: {e}")
    else:
        try:
            from rag_module.search_engine import _ensure_resources, _company_map
            _ensure_resources()
            if _company_map:
                name_to_code = list(_company_map)
                all_tickers = {code.upper() for _, code in _company_map}
        except Exception:
            pass

    _CACHED_NAME_TO_CODE = name_to_code
    _CACHED_ALL_TICKERS = all_tickers
    return _CACHED_NAME_TO_CODE, _CACHED_ALL_TICKERS


def _normalize_company_name(company_input: str, user_query: str) -> str:
    """Normalize company name or raw ticker input against brand aliases and code_stock.csv.

    Resolution Precedence:
    1. Filter out corporate prefix blocklist (CTCP, TMCP, Tập đoàn...).
    2. Check explicit parentheses ticker in query: (TICKER), e.g. (VJC), (NVL), (HHV).
    3. Check ALIAS_TICKER_MAP (sorted by length descending).
    4. Check company names from code_stock.csv (sorted by length descending, including clean names).
    5. Check standalone uppercase 3-5 letter words in query against all_tickers (excluding NEGATIVE_BLOCKLIST).
    6. Validate fallback company_input from LLM.
    """
    company_input = company_input.strip() if company_input else ""
    user_query = user_query.strip() if user_query else ""
    q_lower = user_query.lower()

    if company_input.upper() in NEGATIVE_BLOCKLIST:
        company_input = ""

    name_to_code, all_tickers = _get_stock_mappings()

    # Priority 1: Check explicit parenthesized ticker: (TICKER)
    paren_match = re.search(r"\(([A-Za-z]{2,5})\)", user_query)
    if paren_match:
        paren_code = paren_match.group(1).upper()
        if paren_code in all_tickers and paren_code not in NEGATIVE_BLOCKLIST:
            return paren_code

    # Priority 2: Check ALIAS_TICKER_MAP (longest alias match first)
    sorted_aliases = sorted(ALIAS_TICKER_MAP.items(), key=lambda x: len(x[0]), reverse=True)
    for alias_name, alias_ticker in sorted_aliases:
        if alias_name in q_lower:
            return alias_ticker

    # Priority 3: Check registered company names from code_stock.csv (longest name first)
    sorted_names = sorted(name_to_code, key=lambda x: len(x[0]), reverse=True)
    for name, code in sorted_names:
        if len(name) >= 3 and name.lower() in q_lower:
            return code

    # Priority 4: Scan user_query directly for explicit 3-5 letter uppercase tickers in code_stock.csv
    for w in re.findall(r"\b[A-Za-z]{3,5}\b", user_query):
        w_up = w.upper()
        if w_up in all_tickers and w_up not in NEGATIVE_BLOCKLIST:
            return w_up

    # Priority 5: Validate LLM-supplied company_input
    if company_input:
        c_upper = company_input.upper()
        if c_upper in all_tickers and c_upper not in NEGATIVE_BLOCKLIST:
            if c_upper.lower() in q_lower or c_upper in user_query:
                return c_upper
        c_lower = company_input.lower()
        for name, code in sorted_names:
            if len(name) >= 3 and (name.lower() in c_lower or c_lower in name.lower()):
                if name.lower() in q_lower or code.lower() in q_lower:
                    return code

    if company_input and (company_input.upper() in NEGATIVE_BLOCKLIST or (company_input.lower() not in q_lower and company_input.upper() not in user_query)):
        return ""

    return company_input


def _clean_financial_content(text: str) -> str:
    """Clean action phrases, measurement prefixes, and query noise from financial content string."""
    if not text:
        return ""

    cleaned = text.strip()

    # Strip leading action/measurement phrases
    strip_patterns = [
        r"^tốc\s+độ\s+tăng\s+trưởng\s*%\s*",
        r"^tốc\s+độ\s+tăng\s+trưởng\s*",
        r"^tăng\s+trưởng\s*%\s*",
        r"^tăng\s+trưởng\s*",
        r"^tỷ\s+lệ\s+tăng\s+trưởng\s*",
        r"^mức\s+biến\s+động\s*",
        r"^chênh\s+lệch\s*",
        r"^so\s+sánh\s*",
        r"^tính\s+tổng\s*",
        r"^trích\s+xuất\s*",
        r"^cho\s+biết\s*",
        r"^số\s+dư\s+",
        r"^tổng\s+số\s+",
        r"^tổng\s+giá\s+trị\s+",
        r"^khoản\s+",
        r"^giá\s+trị\s+còn\s+lại\s+của\s+",
    ]

    for pattern in strip_patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    # Strip trailing question/noise phrases
    trailing_patterns = [
        r"\s*là\s+bao\s+nhiêu\??$",
        r"\s*bao\s+nhiêu\??$",
        r"\s*thay\s+đổi\s+như\s+thế\s+nào\??$",
        r"\s*như\s+thế\s+nào\??$",
    ]
    for pattern in trailing_patterns:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    return cleaned.strip()


def extract_secondary_criteria(user_query: str) -> List[str]:
    """Phân tích và trích xuất tất cả tiêu chí phụ (secondary criteria) từ câu hỏi tài chính."""
    if not user_query:
        return []
    
    q_lower = user_query.lower()
    criteria = []

    # 1. Phạm vi báo cáo (Reporting Scope)
    if "công ty mẹ" in q_lower or "cty mẹ" in q_lower:
        criteria.append("công ty mẹ")
    elif "hợp nhất" in q_lower:
        criteria.append("hợp nhất")
    elif "báo cáo riêng" in q_lower or "bctc riêng" in q_lower:
        criteria.append("báo cáo riêng")

    # 2. Mốc thời gian & Cột số liệu (Time & Column Marker)
    m_end_year = re.search(r"(?:cuối năm|kết thúc năm)\s*(\d{4})", q_lower)
    if m_end_year:
        criteria.append(f"cuối năm {m_end_year.group(1)}")
    
    m_date = re.search(r"(?:ngày\s*)?31[/_\s-]12[/_\s-](\d{4})|(?:ngày\s*)?31\s+tháng\s+12\s+năm\s+(\d{4})", q_lower)
    if m_date:
        yr = m_date.group(1) or m_date.group(2)
        criteria.append(f"31/12/{yr}")
        
    m_start_year = re.search(r"đầu năm\s*(\d{4})?", q_lower)
    if m_start_year:
        criteria.append("đầu năm")

    # 3. Tỷ lệ & Phần trăm (Ratios & Percentages)
    if any(k in q_lower for k in ["quyền biểu quyết", "tỷ lệ biểu quyết", "tỷ lệ sở hữu", "lãi suất", "%", "phần trăm"]):
        if "quyền biểu quyết" in q_lower:
            criteria.append("quyền biểu quyết")
        elif "tỷ lệ biểu quyết" in q_lower:
            criteria.append("tỷ lệ biểu quyết")
        elif "tỷ lệ sở hữu" in q_lower:
            criteria.append("tỷ lệ sở hữu")
        else:
            criteria.append("tỷ lệ %")

    # 4. Ngành / Đối tượng / Phân đoạn / Cá nhân (Sector / Breakdown / Entity)
    m_sector = re.search(r"ngành\s+([a-zA-Zà-ỹÀ-Ỹ\s]+?)(?=\s+của|\s+cuối|\s+năm|\s+là|\Z)", q_lower)
    if m_sector:
        criteria.append(f"ngành {m_sector.group(1).strip()}")

    specific_keywords = [
        ("cho thuê khô tàu bay", "khô tàu bay"),
        ("khô tàu bay", "khô tàu bay"),
        ("nhà ga hành khách t2", "Nhà ga T2"),
        ("quỹ bình ổn giá xăng dầu", "Quỹ bình ổn giá xăng dầu"),
        ("trái phiếu đặc biệt", "trái phiếu VAMC"),
        ("vamc", "VAMC"),
        ("hoàng anh gia lai", "Hoàng Anh Gia Lai"),
        ("bảo việt nhân thọ", "Bảo Việt Nhân Thọ"),
        ("visorutex", "Visorutex"),
        ("phát hành thêm", "phát hành thêm"),
        ("góp vốn vào đơn vị khác", "đầu tư đơn vị khác"),
    ]
    for kw, label in specific_keywords:
        if kw in q_lower and label not in criteria:
            criteria.append(label)

    return criteria


def _fallback_parse_query(user_query: str) -> Dict[str, Any]:
    """Fallback rule-based parser when LLM is unreachable."""
    q = user_query.strip()

    # Check for range pattern e.g. "từ năm 2021 đến năm 2023" or "từ 2021 đến 2023"
    range_match = re.search(r"từ\s*(?:năm\s*)?(\d{4})\s*đến\s*(?:năm\s*)?(\d{4})", q, re.IGNORECASE)
    if range_match:
        y1, y2 = int(range_match.group(1)), int(range_match.group(2))
        start_y, end_y = min(y1, y2), max(y1, y2)
        years = [str(y) for y in range(start_y, end_y + 1)]
        tieu_chi_phu = range_match.group(0)
    else:
        years = re.findall(r"\b(20\d{2})\b", q)
        extracted_crit = extract_secondary_criteria(q)
        tieu_chi_phu = ', '.join(extracted_crit) if extracted_crit else None

    q_lower = q.lower()
    if "so sánh" in q_lower or "thay đổi" in q_lower or "tăng trưởng" in q_lower or "từ năm" in q_lower or "đến năm" in q_lower:
        thao_tac = "so_sanh"
    else:
        thao_tac = "trich_xuat"

    # Extract company / ticker
    company = _normalize_company_name("", user_query)

    clean_content = re.sub(r"\b(20\d{2})\b", "", q)
    if company:
        clean_content = re.sub(rf"\b{company}\b", "", clean_content, flags=re.IGNORECASE)

    # Remove growth/comparison stop words
    stop_phrases = [
        "tốc độ tăng trưởng %", "tốc độ tăng trưởng", "tăng trưởng %", "tăng trưởng",
        "so sánh", "từ năm", "đến năm", "của", "năm", "báo cáo", "tài chính",
        "cho", "là bao nhiêu", "bao nhiêu"
    ]
    for word in stop_phrases:
        clean_content = re.sub(rf"\b{re.escape(word)}\b", "", clean_content, flags=re.IGNORECASE)

    # Check for personnel entity in query
    from pipeline.src.nodes.code_generator import extract_person_name
    person_name = extract_person_name(user_query)

    return {
        "ticker": company,
        "ten_cong_ty": company,
        "year": ", ".join(years),
        "so_nam": years,
        "metric": clean_content or q,
        "noi_dung": clean_content or q,
        "thao_tac": thao_tac,
        "muc_tieu": thao_tac,
        "tieu_chi_phu": tieu_chi_phu,
        "ten_nhan_su": person_name,
    }


def parse_query_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 1: Phân tích câu hỏi thành cấu trúc truy vấn.

    Trích xuất: ten_cong_ty, so_nam, noi_dung, thao_tac (muc_tieu: trich_xuat | so_sanh), tieu_chi_phu.
    KHÔNG tìm bảng — Data Discovery sẽ xử lý.

    Args:
        state: Current AgentState containing 'user_query'
        cfg: Config instance (defaults to global config)

    Returns:
        Updated AgentState with 'parsed_query', 'status', and latency tracking.
    """
    cfg = cfg or default_config
    start_time = time.time()
    user_query = state.get("user_query", "").strip()

    if not user_query:
        return {
            **state,
            "status": "error",
            "error_message": "User query is empty.",
            "parsed_query": {},
        }

    try:
        # Load prompts
        prompt_data = load_query_parser_prompt(cfg)
        system_prompt = prompt_data["system_prompt"]
        json_schema = prompt_data["json_schema"]
        few_shots = prompt_data.get("few_shot_examples", [])

        # Build prompt messages
        prompt_messages = [
            SystemMessage(content=f"{system_prompt}\n\nSchema Yêu cầu:\n{json_schema}")
        ]

        for example in few_shots:
            prompt_messages.append(HumanMessage(content=example["user_query"]))
            prompt_messages.append(AIMessage(content=example["parsed_output"]))

        prompt_messages.append(HumanMessage(content=f"Câu hỏi: {user_query}"))

        # Call LLM with slight temperature=0.1 to avoid overfitting/copy-pasting examples
        llm = get_llm(cfg=cfg, temperature=0.1)
        response = llm.invoke(prompt_messages)

        raw_content = response.content if isinstance(response.content, str) else str(response.content)

        # Extract and print agent thoughts
        print(f"\n🔍 [Query Parser] Đang phân tích câu hỏi: '{user_query}'")
        think_match = re.search(r"<think>(.*?)</think>", raw_content, re.DOTALL)
        if think_match:
            thought = think_match.group(1).strip()
            indented_thought = thought.replace("\n", "\n  ")
            print(f"💭 [Tư duy - Query Parser]:\n  {indented_thought}")
        else:
            json_start = raw_content.find("{")
            if json_start > 10:
                thought = raw_content[:json_start].strip()
                indented_thought = thought.replace("\n", "\n  ")
                print(f"💭 [Tư duy - Query Parser]:\n  {indented_thought}")

        # Parse output JSON
        parsed_json = safe_parse_json(raw_content)

        # Map strict schema keys {\"ticker\", \"year\", \"metric\"} to state schema keys
        raw_ticker_val = parsed_json.get("ticker") or parsed_json.get("ten_cong_ty") or ""
        metric_val = parsed_json.get("metric") or parsed_json.get("noi_dung") or ""
        year_val = parsed_json.get("year") if "year" in parsed_json else parsed_json.get("so_nam")

        # Fallback year extraction from query if year is None or empty
        if not year_val or year_val is None or str(year_val).strip() in ["None", "null", ""]:
            year_val = re.findall(r"\b(20\d{2})\b", user_query)

        # Ensure so_nam is a list of strings
        if isinstance(year_val, str):
            so_nam_list = [y.strip() for y in year_val.replace(",", " ").split() if y.strip().isdigit()]
            if not so_nam_list:
                so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)
        elif isinstance(year_val, (int, float)):
            so_nam_list = [str(int(year_val))]
        elif isinstance(year_val, list):
            so_nam_list = [str(y).strip() for y in year_val if str(y).strip().isdigit()]
        else:
            so_nam_list = re.findall(r"\b(20\d{2})\b", user_query)

        # Synchronize ticker / ten_cong_ty strictly
        resolved_ticker = _normalize_company_name(raw_ticker_val, user_query)
        parsed_json["ticker"] = resolved_ticker
        parsed_json["ten_cong_ty"] = resolved_ticker
        parsed_json["year"] = ", ".join(so_nam_list) if so_nam_list else ""
        parsed_json["so_nam"] = so_nam_list

        # Ensure minimal structure and sync thao_tac / muc_tieu (only trich_xuat or so_sanh)
        thao_tac = parsed_json.get("thao_tac") or parsed_json.get("muc_tieu") or (
            "so_sanh" if len(so_nam_list) > 1 or "so sánh" in user_query.lower() or "tăng trưởng" in user_query.lower() else "trich_xuat"
        )
        if thao_tac not in ["trich_xuat", "so_sanh"]:
            thao_tac = "trich_xuat"

        parsed_json["thao_tac"] = thao_tac
        parsed_json["muc_tieu"] = thao_tac

        # Clean metric / noi_dung using _clean_financial_content
        cleaned_content = _clean_financial_content(metric_val) or metric_val
        parsed_json["metric"] = cleaned_content
        parsed_json["noi_dung"] = cleaned_content

        if "tieu_chi_phu" not in parsed_json:
            parsed_json["tieu_chi_phu"] = None

        from pipeline.src.nodes.code_generator import extract_person_name
        person_name = parsed_json.get("ten_nhan_su") or extract_person_name(user_query)
        parsed_json["ten_nhan_su"] = person_name

        print(
            f"📊 [Kết quả - Query Parser]:\n"
            f"   Công ty: {parsed_json.get('ten_cong_ty')}\n"
            f"   Năm: {parsed_json.get('so_nam')}\n"
            f"   Nội dung: {parsed_json.get('noi_dung')}\n"
            f"   Thao tác: {parsed_json.get('thao_tac')}\n"
            f"   Tiêu chí phụ: {parsed_json.get('tieu_chi_phu')}\n"
            f"   Nhân sự: {parsed_json.get('ten_nhan_su')}\n"
        )

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        return {
            **state,
            "parsed_query": parsed_json,
            "status": "pending",
            "node_latencies": node_latencies,
        }

    except Exception as e:
        print(f"⚠️ [Query Parser] LLM không phản hồi ({e}). Đang sử dụng Rule-based Fallback Parser...")
        parsed_json = _fallback_parse_query(user_query)

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["query_parser"] = round(latency, 3)

        print(
            f"📊 [Kết quả Fallback - Query Parser]:\n"
            f"   Công ty: {parsed_json.get('ten_cong_ty')}\n"
            f"   Năm: {parsed_json.get('so_nam')}\n"
            f"   Nội dung: {parsed_json.get('noi_dung')}\n"
            f"   Thao tác: {parsed_json.get('thao_tac')}\n"
            f"   Tiêu chí phụ: {parsed_json.get('tieu_chi_phu')}\n"
        )

        return {
            **state,
            "parsed_query": parsed_json,
            "status": "pending",
            "node_latencies": node_latencies,
        }


### Node 2: Data Discovery Node
Dùng `Search Engine` tra cứu bảng dữ liệu tương ứng theo công ty, năm, nội dung.

In [26]:
"""Node 2: Data Discovery Node.
Tìm kiếm chính xác các bảng dữ liệu bằng Search Engine (search_by_company_and_content)
dựa trên bộ 3 thông tin: ten_cong_ty + so_nam + noi_dung.
"""

import re
import time
import pandas as pd
from pathlib import Path
from typing import Optional, List, Dict, Any

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config
from pipeline.src.utils.data_registry import DataRegistry


# Các cột metadata/thông tin chung - dùng để lọc khi trích xuất first_row_values
_METADATA_COLUMNS = {
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
}


def _extract_table_schema(csv_path: str) -> Dict[str, Any]:
    """Đọc schema (tên cột) và giá trị hàng đầu tiên nếu có cột tên là số.

    Returns:
        Dict với 2 key:
        - table_schema: list[str] — danh sách tên cột
        - first_row_values: dict[str, str] — giá trị hàng đầu tiên (chỉ khi có cột tên số)
    """
    result: Dict[str, Any] = {"table_schema": [], "first_row_values": {}}
    try:
        df = pd.read_csv(csv_path, nrows=1)
        columns = list(df.columns)
        result["table_schema"] = columns

        # Kiểm tra xem có cột nào tên là số (0, 1, 2, 3...) không
        numeric_cols = [c for c in columns if str(c).strip().isdigit()]
        if numeric_cols and not df.empty:
            first_row = df.iloc[0]
            # Trả về giá trị hàng đầu tiên cho các cột không phải metadata
            result["first_row_values"] = {
                str(c): str(first_row[c])
                for c in columns
                if c not in _METADATA_COLUMNS
            }
    except Exception as e:
        print(f"⚠️ Không đọc được schema từ {csv_path}: {e}")
    return result


def _resolve_csv_path(csv_path_str: str, cfg: Config) -> Optional[Path]:
    """Resolve csv_path from search engine result to an actual local file path."""
    if not csv_path_str:
        return None

    p_str = csv_path_str.replace("\\", "/")

    # Try direct path first
    direct = Path(p_str).resolve()
    if direct.exists():
        return direct

    # Try relative from ViFinQA
    idx_fin = p_str.find("ViFinQA")
    if idx_fin != -1:
        relative_part = p_str[idx_fin:]
        repo_root = Path(cfg.DATA_DIR).parent.parent.resolve()

        candidate1 = (repo_root / relative_part).resolve()
        if candidate1.exists():
            return candidate1

        candidate2 = (repo_root / "rag_module" / relative_part).resolve()
        if candidate2.exists():
            return candidate2

    return None


def _get_processed_data_dir(cfg: Config) -> Optional[Path]:
    """Find processed_data directory across local repo and Kaggle environments."""
    candidates = [
        Path(cfg.BASE_DIR.parent / "rag_module" / "ViFinQA" / "processed_data"),
        Path(__file__).resolve().parents[3] / "rag_module" / "ViFinQA" / "processed_data",
        Path("r2AI_2026/rag_module/ViFinQA/processed_data"),
        Path("rag_module/ViFinQA/processed_data"),
        Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data/ViFinQA/processed_data"),
        Path("/kaggle/input/r2-ai-output/r2AI_data/ViFinQA/processed_data"),
    ]
    for c in candidates:
        if c.exists():
            return c.resolve()
    return None


def _find_offline_candidate_tables(
    ticker: str,
    so_nam: List[Any],
    noi_dung: str,
    report_type: Optional[str],
    cfg: Config,
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    """Offline scanner: locate top candidate CSV files directly from local processed_data directory."""
    data_dir = _get_processed_data_dir(cfg)
    if not data_dir or not data_dir.exists():
        return []

    if not ticker:
        return []

    ticker_dir = data_dir / ticker.upper()
    if not ticker_dir.exists() or not ticker_dir.is_dir():
        return []

    year_dirs = []
    if so_nam:
        for y in so_nam:
            yd = ticker_dir / str(y)
            if yd.exists() and yd.is_dir():
                year_dirs.append((str(y), yd))
    if not year_dirs:
        for yd in ticker_dir.iterdir():
            if yd.is_dir():
                year_dirs.append((yd.name, yd))

    tokens = [t.lower() for t in re.findall(r"\w+", noi_dung) if len(t) > 1]
    candidates_with_score = []

    for year_str, yd in year_dirs:
        csv_files = list(yd.glob("**/*.csv"))
        for csv_f in csv_files:
            score = 0.0
            p_lower = str(csv_f).lower()

            if report_type == "consolidated" and "consolidated" in p_lower:
                score += 2.0
            elif report_type == "separate" and "separate" in p_lower:
                score += 2.0

            try:
                sample_df = pd.read_csv(csv_f, nrows=15)
                cols_str = " ".join([str(c) for c in sample_df.columns]).lower()
                cells_str = sample_df.astype(str).to_string().lower()

                for tok in tokens:
                    if tok in cols_str:
                        score += 3.0
                    if tok in cells_str:
                        score += 1.5

                if noi_dung.lower() in cells_str:
                    score += 10.0
            except Exception:
                pass

            candidates_with_score.append((score, year_str, csv_f))

    candidates_with_score.sort(key=lambda x: x[0], reverse=True)

    discovered = []
    seen_paths = set()
    for score, year_str, csv_f in candidates_with_score:
        resolved_p = str(csv_f.resolve())
        if resolved_p not in seen_paths:
            seen_paths.add(resolved_p)
            discovered.append({
                "csv_path": resolved_p,
                "Ten_Bang": csv_f.stem,
                "rrf_score": max(score, 0.1),
                "Ma_Doanh_Nghiep": ticker.upper(),
                "Nam_Tai_Chinh": year_str,
                "Loai_Bao_Cao": report_type or ("consolidated" if "consolidated" in str(csv_f) else "separate"),
            })
            if len(discovered) >= top_k:
                break

    return discovered


def _log_candidates(results: List[Dict[str, Any]], year_label: str = "") -> None:
    """In log chi tiết các ứng viên top-K tìm được từ Search Engine kèm minh chứng khớp cột."""
    prefix = f" (Năm {year_label})" if year_label else ""
    print(f"   📋 Danh sách {len(results)} bảng ứng viên Top-K từ Search Engine{prefix}:")
    for idx, item in enumerate(results, 1):
        p_str = item.get("csv_path", "")
        file_name = Path(p_str).name if p_str else "N/A"
        ten_bang = item.get("Ten_Bang", "N/A")
        rrf = item.get("rrf_score", 0.0)
        dense_r = item.get("dense_rank", "-")
        sparse_r = item.get("sparse_rank", "-")
        matched = item.get("content_matched", False)
        matched_col = item.get("matched_col_name", "")
        matched_sample = item.get("matched_sample", "")

        col_tag = f"Cột '{matched_col}'" if matched_col else "Cột đầu tiên có nghĩa"
        matched_str = f" ✅ [Matched {col_tag}]" if matched else ""
        print(f"      #{idx} RRF: {rrf:.6f} | DenseRank: {dense_r} | SparseRank: {sparse_r}{matched_str}")
        print(f"         File: {file_name}")
        print(f"         Tên bảng: {ten_bang}")
        if matched and matched_sample:
            print(f"         🔍 Minh chứng dòng khớp trong {col_tag}: \"{matched_sample}\"")


def clean_query_content(noi_dung_input: str, ticker: str = "", so_nam: list = None) -> str:
    """Làm sạch chuỗi noi_dung: Loại bỏ tên công ty, mã CK, năm, các từ để hỏi và thông tin thừa
    để đảm bảo Search Engine nhận đúng từ khóa chỉ tiêu cốt lõi (VD: 'Lãi tiền gửi', 'Quỹ khen thưởng, phúc lợi').
    """
    if not noi_dung_input:
        return ""
    text = str(noi_dung_input)
    text = re.sub(r"\([A-Za-z]{2,5}\)", "", text)
    text = re.sub(r"\b20\d{2}\b", "", text)
    if ticker:
        text = re.sub(r"\b" + re.escape(ticker) + r"\b", "", text, flags=re.IGNORECASE)

    patterns = [
        r"là bao nhiêu.*",
        r"bao nhiêu.*",
        r"của công ty mẹ.*",
        r"của ngân hàng.*",
        r"của ctcp.*",
        r"của tập đoàn.*",
        r"của công ty.*",
        r"vào ngày.*",
        r"đến ngày.*",
        r"tại ngày.*",
        r"cuối năm.*",
        r"đầu năm.*",
        r"trong năm.*",
        r"năm.*",
        r"báo cáo tài chính.*",
        r"báo cáo riêng.*",
        r"báo cáo hợp nhất.*",
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE)

    prefix_patterns = [
        r"^\s*tổng\s+số\s+",
        r"^\s*tổng\s+",
        r"^\s*số\s+dư\s+",
        r"^\s*giá\s+trị\s+",
        r"^\s*chỉ\s+tiêu\s+",
    ]
    for pp in prefix_patterns:
        text = re.sub(pp, "", text, flags=re.IGNORECASE)

    cleaned = text.strip(" ,.?:;\t\n")
    return cleaned if len(cleaned) >= 2 else noi_dung_input.strip()


def data_discovery_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 2: Tìm kiếm bảng dữ liệu liên quan bằng Search Engine.

    Sử dụng search_by_company_and_content() dựa trên ten_cong_ty, so_nam, và noi_dung.

    Args:
        state: Current AgentState containing 'parsed_query' and 'user_query'
        cfg: System Config instance

    Returns:
        Updated AgentState with 'discovered_tables' and 'matched_table_path'.
    """
    cfg = cfg or default_config
    start_time = time.time()

    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})

    # Extract fields from parsed_query
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    noi_dung_raw = parsed_query.get("noi_dung", "")
    thao_tac = parsed_query.get("thao_tac") or parsed_query.get("muc_tieu", "trich_xuat")

    if not so_nam and isinstance(user_query, str):
        so_nam = re.findall(r"\b(20\d{2})\b", user_query)
    if isinstance(user_query, str) and not ten_cong_ty:
        m_ticker = re.search(r"\b([A-Za-z]{3,5})\b", user_query)
        if m_ticker:
            ten_cong_ty = m_ticker.group(1).upper()

    if not noi_dung_raw:
        noi_dung_raw = user_query if isinstance(user_query, str) else ""

    # Làm sạch noi_dung trước khi truyền vào Search Engine
    noi_dung = clean_query_content(noi_dung_raw, ten_cong_ty, so_nam)

    print(f"\n🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...")
    print(f"   - Công ty: '{ten_cong_ty}'")
    print(f"   - Số năm: {so_nam}")
    print(f"   - Nội dung cần tìm (đã làm sạch): '{noi_dung}' (gốc: '{noi_dung_raw}')")
    print(f"   - Thao tác: {thao_tac}")

    # Determine report type: mặc định None để tìm kiếm trên CẢ 2 loại báo cáo (consolidated & separate)
    report_type = None
    if isinstance(user_query, str):
        q_lower = user_query.lower()
        if "hợp nhất" in q_lower and "riêng" not in q_lower:
            report_type = "consolidated"
        elif "báo cáo riêng" in q_lower:
            report_type = "separate"

    all_discovered_tables: List[Dict[str, Any]] = []

    # Import Search Engine
    try:
        from rag_module.search_engine import search_by_company_and_content
        import rag_module.search_engine as se

        se._ensure_resources()

        if not so_nam:
            print(f"   - Tra cứu bảng cho công ty '{ten_cong_ty}' và nội dung '{noi_dung}'...")
            results = search_by_company_and_content(
                company_name=ten_cong_ty,
                content=noi_dung,
                raw_query=user_query,
                year=None,
                report_type=report_type,
                top_k=5,
            )
            if not results and report_type is not None:
                results = search_by_company_and_content(
                    company_name=ten_cong_ty,
                    content=noi_dung,
                    raw_query=user_query,
                    year=None,
                    report_type=None,
                    top_k=5,
                )
            if results:
                _log_candidates(results)
                for match in results[:5]:
                    csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                    if csv_path:
                        table_entry = {
                            "csv_path": str(csv_path),
                            "Ten_Bang": match.get("Ten_Bang", ""),
                            "rrf_score": match.get("rrf_score", 0.0),
                            "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                            "Nam_Tai_Chinh": match.get("Nam_Tai_Chinh", ""),
                            "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                            "matched_sample": match.get("matched_sample", ""),
                        }
                        if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                            all_discovered_tables.append(table_entry)
                if all_discovered_tables:
                    best = all_discovered_tables[0]
                    print(f"   🏆 Bảng khớp CAO NHẤT: {Path(best['csv_path']).name} — {best['Ten_Bang']} (RRF: {best['rrf_score']:.6f})")
        else:
            for year in so_nam:
                print(f"   - Tra cứu bảng cho năm {year}...")
                results = search_by_company_and_content(
                    company_name=ten_cong_ty,
                    content=noi_dung,
                    raw_query=user_query,
                    year=str(year),
                    report_type=report_type,
                    top_k=5,
                )
                if not results and report_type is not None:
                    results = search_by_company_and_content(
                        company_name=ten_cong_ty,
                        content=noi_dung,
                        raw_query=user_query,
                        year=str(year),
                        report_type=None,
                        top_k=5,
                    )
                if results:
                    _log_candidates(results, year_label=str(year))
                    for match in results[:5]:
                        csv_path = _resolve_csv_path(match.get("csv_path", ""), cfg)
                        if csv_path:
                            table_entry = {
                                "csv_path": str(csv_path),
                                "Ten_Bang": match.get("Ten_Bang", ""),
                                "rrf_score": match.get("rrf_score", 0.0),
                                "Ma_Doanh_Nghiep": match.get("Ma_Doanh_Nghiep", ten_cong_ty),
                                "Nam_Tai_Chinh": str(year),
                                "Loai_Bao_Cao": match.get("Loai_Bao_Cao", ""),
                                "matched_sample": match.get("matched_sample", ""),
                            }
                            if not any(t["csv_path"] == str(csv_path) for t in all_discovered_tables):
                                all_discovered_tables.append(table_entry)
                    if all_discovered_tables:
                        best = all_discovered_tables[0]
                        print(f"   🏆 Năm {year} - Bảng khớp CAO NHẤT: {Path(best['csv_path']).name} — {best['Ten_Bang']} (RRF: {best['rrf_score']:.6f})")

    except Exception as e:
        print(f"⚠️ [Data Discovery] Lỗi/Không dùng được Search Engine: {e}. Thử quét thư mục offline...")

    # Fallback 1: Offline directory scanner when Search Engine is offline/unavailable
    if not all_discovered_tables and ten_cong_ty:
        try:
            offline_tables = _find_offline_candidate_tables(
                ticker=ten_cong_ty,
                so_nam=so_nam,
                noi_dung=noi_dung,
                report_type=report_type,
                cfg=cfg,
                top_k=5,
            )
            for tbl in offline_tables:
                if not any(t["csv_path"] == tbl["csv_path"] for t in all_discovered_tables):
                    all_discovered_tables.append(tbl)
            if all_discovered_tables:
                print(f"   📂 [Data Discovery] Đã nạp thành công {len(all_discovered_tables)} bảng từ thư mục processed_data offline.")
        except Exception as scan_err:
            print(f"⚠️ [Data Discovery] Offline scanner error: {scan_err}")

    # Fallback 2: DataRegistry if still nothing found
    if not all_discovered_tables:
        try:
            registry = DataRegistry(cfg=cfg)
            matched_path = registry.find_best_match(noi_dung) or registry.find_best_match(ten_cong_ty)
            if matched_path and matched_path.exists():
                all_discovered_tables.append({
                    "csv_path": str(matched_path),
                    "Ten_Bang": matched_path.stem,
                    "rrf_score": 1.0,
                    "Ma_Doanh_Nghiep": ten_cong_ty,
                    "Nam_Tai_Chinh": so_nam[0] if so_nam else "",
                    "Loai_Bao_Cao": report_type,
                })
        except Exception as reg_err:
            print(f"⚠️ [Data Discovery] Registry fallback error: {reg_err}")

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["data_discovery"] = round(latency, 3)

    if not all_discovered_tables:
        print(f"\n❌ [Data Discovery] Không tìm thấy bảng dữ liệu nào phù hợp.")
        return {
            **state,
            "status": "error",
            "error_message": "Không tìm thấy bảng dữ liệu phù hợp từ Search Engine.",
            "discovered_tables": [],
            "top_k_candidates": [],
            "matched_table_path": None,
            "table_schema": [],
            "first_row_values": {},
            "node_latencies": node_latencies,
        }

    # Extract schema và first_row_values cho tất cả các bảng trong Top 5
    for tbl in all_discovered_tables[:5]:
        schema_info = _extract_table_schema(tbl["csv_path"])
        tbl["table_schema"] = schema_info["table_schema"]
        tbl["first_row_values"] = schema_info["first_row_values"]

    first_table_path = all_discovered_tables[0]["csv_path"]
    first_schema = all_discovered_tables[0].get("table_schema", [])
    first_row_val = all_discovered_tables[0].get("first_row_values", {})

    print(f"\n📊 [Kết quả - Data Discovery]: Đã chọn {len(all_discovered_tables)} bảng có độ khớp cao nhất (Top-K candidates: {min(5, len(all_discovered_tables))}).")
    print(f"   📋 Schema bảng Top 1: {first_schema}")
    if first_row_val:
        print(f"   📋 Giá trị hàng đầu tiên (cột số): {first_row_val}")
    print()

    return {
        **state,
        "discovered_tables": all_discovered_tables,
        "top_k_candidates": all_discovered_tables[:5],
        "matched_table_path": first_table_path,
        "table_schema": first_schema,
        "first_row_values": first_row_val,
        "status": "pending",
        "node_latencies": node_latencies,
    }


print('✅ Node 2: Data Discovery Node loaded!')


✅ Node 2: Data Discovery Node loaded!


### Node 3: Schema Mapper Node
Duyệt từng cột trong bảng dữ liệu thực tế để xác thực dữ liệu:
1. Truy vấn các cột từ bảng (Data Table).
2. Xác nhận: Số hàng có dữ liệu > Số hàng không có dữ liệu trong cột (loại bỏ cột rỗng/sparse).
3. Lựa chọn các cột thỏa mãn là Useful Columns.
4. Tìm tên cột thực sự (Header Resolution khi tên cột là số/unnamed).
5. Đưa ra mô tả ngữ nghĩa cho cột dựa vào tên và nội dung.
6. Xác định động cột nhãn (label_column) và cột giá trị (value_column).
7. Xuất và in ra Output JSON chuẩn phục vụ kiểm thử.

In [27]:
"""Node 3: Schema Mapper Node.
Duyệt từng cột trong bảng dữ liệu thực tế để xác thực dữ liệu:
Workflow:
1. Truy vấn các cột từ bảng (Data Table)
2. Xác nhận: Số hàng có dữ liệu > Số hàng không có dữ liệu trong cột
3. Lựa chọn các cột thỏa mãn là Useful Columns
4. Tìm tên cột thực sự (Header Resolution khi tên cột là số/unnamed)
5. Đưa ra mô tả ngữ nghĩa cho cột dựa vào tên và nội dung
6. Xác định động cột nhãn (label_column) và cột giá trị (value_column)
7. Xuất và in ra Output JSON chuẩn phục vụ kiểm thử.
"""

import re
import time
import json
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional, Set

from thefuzz import process, fuzz
from langchain_core.messages import SystemMessage, HumanMessage

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config
from pipeline.src.llm_provider import get_llm
from pipeline.src.utils.json_repair import safe_parse_json


# Các cột metadata/thông tin chung ở đầu file CSV cần bỏ qua khi phân tích dữ liệu bảng
METADATA_HEADER_COLUMNS = [
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
]


AUXILIARY_COL_REGEX = re.compile(
    r"^(stt|số\s*tt|số\s*thứ\s*tự|sothutu|mã\s*số|mãsố|thuyết\s*minh|thuyếtminh|ghi\s*chú|note|code|ms|tm|cột_\d+|unnamed.*)$",
    re.IGNORECASE,
)

_AUXILIARY_CODE_COLUMNS = {
    "mã số", "mãsố", "thuyết minh", "thuyếtminh", "stt", "số tt", "số thứ tự", "sothutu",
    "ghi chú", "note", "code", "ms", "tm", "cột_0", "cột 0", "cot_0", "cot 0", "unnamed: 0"
}


def _is_cell_empty(val: Any) -> bool:
    """Kiểm tra xem một ô có bị rỗng, null hoặc chứa ký tự không có dữ liệu hay không."""
    if val is None or pd.isna(val):
        return True
    s = str(val).strip()
    return s.lower() in {"", "nan", "none", "null", "n/a", "na", "-", "—", "--", "nil"}


def _is_numeric_value(val: Any) -> bool:
    """Kiểm tra xem một ô có chứa giá trị số (kể cả số âm trong ngoặc, dấu phân cách, %, $, VND) hay không."""
    if _is_cell_empty(val):
        return False
    s = str(val).strip()
    s = re.sub(r"(?i)\b(?:vnd|đồng|dong|usd)\b", "", s).strip()
    s = s.replace(",", "").replace(".", "").replace(" ", "").replace("%", "").replace("$", "").replace("VND", "").replace("vnd", "")
    if s.startswith("(") and s.endswith(")"):
        s = s[1:-1]
    if s.startswith("-") or s.startswith("+"):
        s = s[1:]
    return s.isdigit() and len(s) > 0


def _resolve_column_header(df: pd.DataFrame, col: str) -> str:
    """Xác định tên thực sự của cột.

    - Nếu tên cột là năm 4 chữ số (ví dụ: '2018', '2017') -> giữ nguyên.
    - Nếu tên cột là chữ có nghĩa -> giữ nguyên.
    - Nếu tên cột là số thứ tự positional ('0', '1', '2',...) hoặc 'Unnamed:' -> quét các hàng đầu tiên để tìm tiêu đề chữ.
    """
    raw_name = str(col).strip()

    # 1. Nếu là năm 4 chữ số (19xx, 20xx) -> là header hợp lệ, giữ nguyên!
    if re.match(r"^(19|20)\d{2}$", raw_name):
        return raw_name

    # 2. Nếu là tên rõ ràng (chứa chữ và không phải 'Unnamed:' hoặc positional số) -> giữ nguyên!
    if not raw_name.isdigit() and not raw_name.startswith("Unnamed:"):
        return raw_name

    # 3. Quét các hàng header đầu tiên để tìm chuỗi văn bản tiêu đề
    for row_idx in range(min(5, len(df))):
        cell_val = df.iloc[row_idx][col]
        if not _is_cell_empty(cell_val):
            cell_str = str(cell_val).strip()
            if not _is_numeric_value(cell_str):
                has_letters = bool(re.search(r"[a-zA-ZÀ-ỹ]", cell_str))
                has_date_format = bool(re.search(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", cell_str))
                if has_letters or has_date_format:
                    return cell_str

    if raw_name == "0":
        return "Chỉ tiêu"

    return raw_name


def _clean_table_name(table_name: str) -> str:
    """Làm sạch tên bảng báo cáo tài chính để phục vụ việc mô tả cột."""
    if not table_name:
        return "Báo cáo tài chính"
    text = str(table_name).strip()
    text = re.sub(r"^(\d+\s*[\.\)]|\w+\s*[\.\)]|\*\))\s*", "", text)
    patterns_to_remove = [
        r"Mẫu\s+B\s*\d+\s*-\s*DN.*",
        r"Ban\s+hành\s+theo\s+Thông\s+tư.*",
        r"Cho\s+năm\s+tài\s+chính\s+kết\s+thúc.*",
        r"cho\s+năm\s+kết\s+thúc\s+ngày.*",
        r"kết\s+thúc\s+ngày\s+\d+.*",
        r"vào\s+ngày\s+\d+.*",
        r"Tại\s+ngày\s+\d+.*",
        r"\(?\s*tiếp\s+theo.*",
        r"_\s*Phải\s+trả.*",
    ]
    for p in patterns_to_remove:
        text = re.sub(p, "", text, flags=re.IGNORECASE).strip()
    text = text.strip(" :,-_.\t\n()[]{}")
    return text if len(text) >= 3 else table_name.strip()


def _extract_unit_from_name(name: str, default_unit: Optional[str] = None) -> Optional[str]:
    """Phát hiện đơn vị tính/đo lường từ tên cột hoặc metadata để làm rõ trong mô tả."""
    lower = name.lower()
    if "triệu" in lower or "trieu" in lower or "million" in lower:
        return "triệu đồng"
    if "tỷ" in lower or "ty" in lower or "billion" in lower:
        return "tỷ đồng"
    if "nghìn" in lower or "ngàn" in lower or "thousand" in lower:
        return "nghìn đồng"
    if "usd" in lower or "$" in lower:
        return "USD"
    if "vnd" in lower or "đồng" in lower:
        return "VND"
    if default_unit:
        d_lower = str(default_unit).lower().strip()
        if d_lower and d_lower not in {"vnd", "đồng", "dong", "none", "nan", "-"}:
            return str(default_unit).strip()
    return None


def _build_default_column_description(
    table_name: str, 
    col_name: str, 
    raw_column: str,
    don_vi_tinh: Optional[str] = None,
) -> str:
    """Tạo mô tả chi tiết mặc định kết hợp ngữ cảnh tên bảng, thời kỳ và đơn vị tính của cột."""
    clean_tbl = _clean_table_name(table_name)
    name_check = f"{col_name} {raw_column}".strip()
    unit = _extract_unit_from_name(name_check, don_vi_tinh)
    unit_suffix = f" (đơn vị đo: {unit})" if unit else ""

    # 1. Định dạng ngày cụ thể (ví dụ: 31/12/2024, 01/01/2023)
    date_match = re.search(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", name_check)
    if date_match:
        date_str = date_match.group(0)
        return f"{clean_tbl} của các nội dung tại ngày {date_str}{unit_suffix}"

    # 2. Tìm năm 4 chữ số trong tên cột (ví dụ: '2018', '2017', 'Năm 2020Triệu', '2021Triệu', '2020 VND')
    year_match = re.search(r"(19\d{2}|20\d{2})", name_check)
    if year_match:
        year = year_match.group(1)
        return f"{clean_tbl} của các nội dung trong năm {year}{unit_suffix}"

    # 3. Các từ khóa chỉ kỳ / thời điểm tương đối
    lower_check = name_check.lower()
    if "năm nay" in lower_check or "kỳ này" in lower_check:
        return f"{clean_tbl} của các nội dung trong năm nay / kỳ hiện tại{unit_suffix}"
    if "năm trước" in lower_check or "kỳ trước" in lower_check:
        return f"{clean_tbl} của các nội dung trong năm trước / kỳ trước{unit_suffix}"
    if "cuối năm" in lower_check or "cuối kỳ" in lower_check:
        return f"{clean_tbl} của các nội dung vào thời điểm cuối năm / cuối kỳ{unit_suffix}"
    if "đầu năm" in lower_check or "đầu kỳ" in lower_check:
        return f"{clean_tbl} của các nội dung vào thời điểm đầu năm / đầu kỳ{unit_suffix}"

    # 4. Fallback chung
    display_col = col_name if col_name and not col_name.isdigit() else raw_column
    return f"{clean_tbl} - chỉ tiêu {display_col}{unit_suffix}"


def _get_columns_from_table(table: Dict[str, Any]) -> List[str]:
    """Extract column names from a discovered table by reading its CSV file."""
    csv_path = table.get("csv_path", "")
    if not csv_path:
        return []

    try:
        df = pd.read_csv(csv_path, nrows=2)
        return list(df.columns)
    except Exception:
        return []


# ─────────────────────────────────────────────────────────────
# Workflow Steps: Useful Columns & Dynamic Mapping
# ─────────────────────────────────────────────────────────────

def _is_code_or_index_column(series: pd.Series, col_name: str = "") -> bool:
    """Kiểm tra xem một cột có phải STT, mã số, thuyết minh, float index hoặc chỉ số phân mục hay không."""
    raw_col_lower = str(col_name).strip().lower()
    if AUXILIARY_COL_REGEX.match(raw_col_lower) or raw_col_lower in _AUXILIARY_CODE_COLUMNS:
        return True

    non_empty = [str(x).strip() for x in series if not _is_cell_empty(x)]
    if not non_empty:
        return False

    total_chars = sum(len(x) for x in non_empty)
    letter_count = sum(sum(1 for ch in x if ch.isalpha()) for x in non_empty)
    avg_len = total_chars / len(non_empty)
    letter_ratio = (letter_count / total_chars) if total_chars > 0 else 0.0

    # Quy tắc 1: Text density check - Chuỗi cực ngắn (<= 4.0 ký tự) và ít chữ cái (< 0.35) -> Cột chỉ số / STT
    if avg_len <= 4.0 and letter_ratio < 0.35:
        return True

    # Quy tắc 2: Khớp các mẫu mã số, số thứ tự, float index (\d+\.0+), số La Mã, phân mục
    index_token_pattern = re.compile(
        r"^(?:[0-9]{1,4}[a-z]?|[IVXLCDM]+|[A-Z]|\(\w+\)|\d+\.\d{1,2}|\d+[\.\)]|\d+\.\d+\.\d+)$",
        re.IGNORECASE,
    )
    index_matches = sum(
        1 for s in non_empty
        if (len(s) <= 4 and not _is_numeric_value(s)) or
           (len(s) <= 5 and bool(re.match(r"^\d+\.0+$", s))) or
           (len(s) <= 6 and bool(index_token_pattern.match(s)))
    )
    return (index_matches / len(non_empty)) >= 0.5


def _extract_useful_columns(
    df: pd.DataFrame,
    label_col: Optional[str] = None,
    metadata_cols: Optional[Set[str]] = None,
) -> List[Dict[str, Any]]:
    """Duyệt từng cột một trong bảng và xác nhận:
    Số hàng có dữ liệu > Số hàng không có dữ liệu.

    Returns:
        List[Dict] thông tin các useful columns.
    """
    metadata_set = metadata_cols or set(METADATA_HEADER_COLUMNS)
    useful = []
    total_rows = len(df)
    candidate_cols = [c for c in df.columns if c not in metadata_set]

    print(f"   📊 [Workflow Step 1: Truy vấn các cột]")
    print(f"      • Tổng số hàng trong bảng: {total_rows}")
    print(f"      • Các cột dữ liệu cần duyệt ({len(candidate_cols)} cột): {candidate_cols}")

    print(f"\n   🔍 [Workflow Step 2: Xác nhận số hàng có dữ liệu > số hàng không có dữ liệu]")
    for col in candidate_cols:
        series = df[col]
        data_rows = 0
        empty_rows = 0
        numeric_count = 0
        sample_values = []
        non_empty_values = []

        for val in series:
            if _is_cell_empty(val):
                empty_rows += 1
            else:
                data_rows += 1
                s_val = str(val).strip()
                non_empty_values.append(s_val)
                if _is_numeric_value(val):
                    numeric_count += 1
                if len(sample_values) < 3:
                    sample_values.append(s_val)

        # Điều kiện bắt buộc: Số hàng có dữ liệu > Số hàng không có dữ liệu
        is_useful = data_rows > empty_rows

        if is_useful:
            print(f"      ✅ Cột '{col}': Có dữ liệu = {data_rows}/{total_rows} > Trống = {empty_rows}/{total_rows} -> HỢP LỆ (Useful)")

            # Step 4: Tìm tên cột thực tế
            resolved_name = _resolve_column_header(df, col)

            total_chars = sum(len(x) for x in non_empty_values)
            letter_count = sum(sum(1 for ch in x if ch.isalpha()) for x in non_empty_values)
            avg_str_len = (total_chars / len(non_empty_values)) if non_empty_values else 0.0
            letter_ratio = (letter_count / total_chars) if total_chars > 0 else 0.0

            # Phân loại auxiliary code column (Mã số, Thuyết minh, STT, Roman numerals, short index codes, float index)
            is_aux_code = (
                bool(AUXILIARY_COL_REGEX.match(resolved_name.strip()))
                or bool(AUXILIARY_COL_REGEX.match(str(col).strip()))
                or resolved_name.strip().lower() in _AUXILIARY_CODE_COLUMNS 
                or str(col).strip().lower() in _AUXILIARY_CODE_COLUMNS
                or _is_code_or_index_column(series, col_name=str(col))
                or _is_code_or_index_column(series, col_name=resolved_name)
            )

            if is_aux_code:
                data_type = "text"
            else:
                data_type = "numeric" if (data_rows > 0 and numeric_count / data_rows >= 0.5) else "text"

            useful.append({
                "raw_column": str(col),
                "column_name": resolved_name,
                "column_index": list(df.columns).index(col),
                "data_type": data_type,
                "is_aux_code": is_aux_code,
                "avg_str_len": avg_str_len,
                "letter_ratio": letter_ratio,
                "data_rows_count": data_rows,
                "empty_rows_count": empty_rows,
                "sample_values": sample_values,
                "column_description": "",
            })
        else:
            print(f"      ❌ Cột '{col}': Có dữ liệu = {data_rows}/{total_rows} <= Trống = {empty_rows}/{total_rows} -> BỎ QUA (Không đủ dữ liệu)")

    return useful


def _find_label_column(
    useful_columns: List[Dict[str, Any]],
    columns: Optional[List[str]] = None
) -> Optional[str]:
    """Xác định cột nhãn (chứa tên chỉ tiêu tài chính) một cách động:
    Chọn cột dạng 'text' có độ dài chuỗi trung bình lớn nhất và mật độ chữ cao nhất.
    """
    if not useful_columns:
        if columns:
            non_meta = [c for c in columns if c not in METADATA_HEADER_COLUMNS]
            return non_meta[0] if non_meta else columns[0]
        return None

    # 1. Ứng viên ưu tiên: Cột text không phải auxiliary code
    primary_text = [
        c for c in useful_columns 
        if c.get("data_type") == "text" 
        and not c.get("is_aux_code", False)
        and not AUXILIARY_COL_REGEX.match(str(c.get("column_name", "")).strip())
        and not AUXILIARY_COL_REGEX.match(str(c.get("raw_column", "")).strip())
        and str(c.get("column_name", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
        and str(c.get("raw_column", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
    ]

    if primary_text:
        # Chọn cột có letter_ratio >= 0.40 và avg_str_len lớn nhất
        high_letter_candidates = [c for c in primary_text if c.get("letter_ratio", 0.0) >= 0.40]
        if high_letter_candidates:
            return max(high_letter_candidates, key=lambda c: c.get("avg_str_len", 0.0))["raw_column"]
        return max(primary_text, key=lambda c: c.get("avg_str_len", 0.0))["raw_column"]

    # 2. Fallback sang bất kỳ cột text nào có avg_str_len lớn nhất
    text_cols = [c for c in useful_columns if c.get("data_type") == "text"]
    if text_cols:
        return max(text_cols, key=lambda c: c.get("avg_str_len", 0.0))["raw_column"]

    # 3. Fallback sang cột đầu tiên trong useful_columns
    return useful_columns[0]["raw_column"]


def _find_value_column(
    useful_columns: List[Dict[str, Any]],
    label_col: Optional[str] = None,
    tieu_chi_phu: Optional[str] = None,
    columns: Optional[List[str]] = None,
) -> Optional[str]:
    """Xác định cột giá trị một cách động dựa trên tiêu chí phụ, cột số và cột phần trăm (%)."""
    if not useful_columns:
        if columns:
            candidates = [c for c in columns if c not in METADATA_HEADER_COLUMNS and c != label_col]
            return candidates[0] if candidates else None
        return None

    value_candidates = [c for c in useful_columns if c.get("raw_column") != label_col]
    if not value_candidates:
        value_candidates = useful_columns

    if tieu_chi_phu and value_candidates:
        clean_tcp = str(tieu_chi_phu).strip().lower()

        # 1. Exact or substring match (An toàn với mọi kiểu dữ liệu tên cột)
        for uc in value_candidates:
            c_name = str(uc.get("column_name", "")).lower()
            r_name = str(uc.get("raw_column", "")).lower()
            c_desc = str(uc.get("column_description", "")).lower()
            if clean_tcp in c_name or clean_tcp in r_name or clean_tcp in c_desc:
                return uc["raw_column"]

        # 2. Khớp chuyên biệt cho truy vấn tỷ lệ / phần trăm (%)
        if any(pct_kw in clean_tcp for pct_kw in ["%", "phần trăm", "tỷ lệ", "ty le", "biểu quyết", "sở hữu", "lãi suất"]):
            for uc in value_candidates:
                c_name = str(uc.get("column_name", "")).lower()
                r_name = str(uc.get("raw_column", "")).lower()
                c_desc = str(uc.get("column_description", "")).lower()
                if any(k in c_name or k in r_name or k in c_desc for k in ["%", "tỷ lệ", "ty le", "biểu quyết", "sở hữu", "lãi suất"]):
                    return uc["raw_column"]

        # 3. Fuzzy match tiêu chí phụ với các tên cột ứng viên
        candidate_names = [str(uc.get("column_name", "")) for uc in value_candidates]
        if candidate_names:
            match, score = process.extractOne(
                clean_tcp, candidate_names, scorer=fuzz.token_set_ratio
            )
            if score >= 50:
                for uc in value_candidates:
                    if str(uc.get("column_name", "")) == match:
                        return uc["raw_column"]

    # Ưu tiên các cột numeric KHÔNG phải là cột mã số / thuyết minh
    primary_numeric = [
        c for c in value_candidates 
        if c.get("data_type") == "numeric" 
        and not c.get("is_aux_code", False)
        and not AUXILIARY_COL_REGEX.match(str(c.get("column_name", "")).strip())
        and not AUXILIARY_COL_REGEX.match(str(c.get("raw_column", "")).strip())
        and str(c.get("column_name", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
        and str(c.get("raw_column", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
    ]
    if primary_numeric:
        return primary_numeric[0]["raw_column"]

    # Default: Chọn cột dạng 'numeric' đầu tiên trong các cột ứng viên
    numeric_candidates = [c for c in value_candidates if c.get("data_type") == "numeric"]
    if numeric_candidates:
        return numeric_candidates[0]["raw_column"]

    return value_candidates[0]["raw_column"]


def _extract_sub_sections(
    df: pd.DataFrame,
    label_col: Optional[str],
    metadata_cols: Set[str],
) -> List[Dict[str, Any]]:
    """Phát hiện các danh mục con (sub-sections) trong bảng tài chính."""
    if not label_col or label_col not in df.columns:
        return []

    value_cols = [c for c in df.columns if c not in metadata_cols and c != label_col]
    if not value_cols:
        return []

    sections = []
    section_header_rows = []

    prev_was_empty = False
    for idx in range(len(df)):
        label_val = df.iloc[idx][label_col]
        is_label_empty = pd.isna(label_val) or str(label_val).strip() == ""

        if not is_label_empty and prev_was_empty:
            label_text = str(label_val).strip()

            total_value = None
            for vc in value_cols:
                cell = df.iloc[idx][vc]
                if pd.notna(cell):
                    try:
                        cell_str = str(cell).strip().replace(",", "")
                        if cell_str.startswith("(") and cell_str.endswith(")"):
                            cell_str = "-" + cell_str[1:-1].strip()
                        total_value = float(cell_str)
                        break
                    except (ValueError, TypeError):
                        continue

            section_header_rows.append({
                "row_idx": idx,
                "section_name": label_text,
                "total_value": total_value,
            })

        prev_was_empty = is_label_empty

    for i, header in enumerate(section_header_rows):
        start = header["row_idx"] + 1
        if i + 1 < len(section_header_rows):
            end = section_header_rows[i + 1]["row_idx"] - 1
        else:
            end = len(df) - 1

        while end >= start:
            val = df.iloc[end][label_col]
            if pd.isna(val) or str(val).strip() == "":
                end -= 1
            else:
                break

        if start <= end:
            sections.append({
                "section_name": header["section_name"],
                "range": [start, end],
                "total_value": header["total_value"],
            })

    return sections


def _enrich_column_descriptions(
    cfg: Config,
    useful_columns: List[Dict[str, Any]],
    table_name: str,
    don_vi_tinh: Optional[str] = None,
) -> List[Dict[str, Any]]:
    """Gán mô tả cấu trúc mặc định cho các cột useful KHÔNG gọi LLM (xử lý tức thì)."""
    if not useful_columns:
        return useful_columns

    for col in useful_columns:
        c_name = col.get("column_name", "")
        r_name = col.get("raw_column", "")
        if not col.get("column_description"):
            col["column_description"] = _build_default_column_description(table_name, c_name, r_name, don_vi_tinh)

    return useful_columns


def analyze_single_table_schema(
    table_dict: Dict[str, Any],
    tieu_chi_phu: Optional[str] = None,
    cfg: Optional[Config] = None,
) -> Dict[str, Any]:
    """Phân tích schema, label_column, value_column cho một bảng bất kỳ."""
    cfg = cfg or default_config
    csv_path = table_dict.get("csv_path", "")
    if not csv_path or not Path(csv_path).exists():
        return {"column_mapping": {}, "schema": {}}

    metadata_set = set(METADATA_HEADER_COLUMNS)
    table_name = table_dict.get("Ten_Bang", Path(csv_path).stem)
    try:
        df_full = pd.read_csv(csv_path)
        raw_columns = list(df_full.columns)
        all_useful = _extract_useful_columns(df_full, metadata_cols=metadata_set)
        label_col = _find_label_column(all_useful, raw_columns)
        useful_cols = [
            c for c in all_useful
            if c.get("data_type") == "numeric"
            and not c.get("is_aux_code", False)
            and not AUXILIARY_COL_REGEX.match(str(c.get("column_name", "")).strip())
            and not AUXILIARY_COL_REGEX.match(str(c.get("raw_column", "")).strip())
            and str(c.get("column_name", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
            and str(c.get("raw_column", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
        ]
        if not useful_cols:
            useful_cols = [c for c in all_useful if c.get("raw_column") != label_col]
        value_col = _find_value_column(useful_cols, label_col, tieu_chi_phu, raw_columns)
        label_col_idx = raw_columns.index(label_col) if (label_col and label_col in raw_columns) else 0
        value_col_idx = raw_columns.index(value_col) if (value_col and value_col in raw_columns) else 1
        return {
            "column_mapping": {
                "label_column": label_col,
                "value_column": value_col,
                "label_column_idx": label_col_idx,
                "value_column_idx": value_col_idx,
                "all_columns": str(raw_columns),
                "all_useful_columns": [c["raw_column"] for c in useful_cols],
            },
            "schema": {
                "useful_columns": useful_cols,
                "sub_sections": _extract_sub_sections(df_full, label_col, metadata_set),
            }
        }
    except Exception as e:
        return {"column_mapping": {}, "schema": {}}


def map_multi_tables(
    tables: List[Dict[str, Any]],
    parsed_query: Dict[str, Any],
    cfg: Optional[Config] = None,
) -> List[Dict[str, Any]]:
    """Enrich candidate tables with schema mappings, label/value columns, and table summaries.

    Args:
        tables: List of candidate table dicts (each having 'csv_path', 'Ten_Bang', etc.)
        parsed_query: Parsed query dictionary (containing 'tieu_chi_phu', 'noi_dung', etc.)
        cfg: Configuration instance

    Returns:
        Enriched list of table dicts, each updated with:
        - 'column_mapping': {'label_column': ..., 'value_column': ..., 'all_columns': ..., 'all_useful_columns': ...}
        - 'schema': {'useful_columns': [...], 'sub_sections': [...]}
        - 'useful_columns': List of useful column names or dicts
        - 'label_column': str
        - 'value_column': str
        - 'table_summary': str
    """
    cfg = cfg or default_config
    tieu_chi_phu = parsed_query.get("tieu_chi_phu") if isinstance(parsed_query, dict) else None
    enriched_tables = []

    for tbl in tables:
        tbl_copy = dict(tbl)
        tbl_res = analyze_single_table_schema(tbl_copy, tieu_chi_phu=tieu_chi_phu, cfg=cfg)
        col_mapping = tbl_res.get("column_mapping", {})
        tbl_schema = tbl_res.get("schema", {})

        label_col = col_mapping.get("label_column", "")
        value_col = col_mapping.get("value_column", "")
        useful_cols = tbl_schema.get("useful_columns", [])
        csv_p = tbl_copy.get("csv_path", "")
        table_name = tbl_copy.get("Ten_Bang", Path(csv_p).stem if csv_p else "")

        tbl_copy["column_mapping"] = col_mapping
        tbl_copy["schema"] = tbl_schema
        tbl_copy["label_column"] = label_col
        tbl_copy["value_column"] = value_col
        tbl_copy["useful_columns"] = useful_cols
        tbl_copy["table_summary"] = f"Bảng: {table_name} | Cột nhãn: {label_col} | Cột giá trị: {value_col}"

        enriched_tables.append(tbl_copy)

    return enriched_tables


def schema_mapper_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 3: Schema Mapper Node.

    Thực hiện quy trình:
    1. Truy vấn các cột từ bảng
    2. Xác nhận số hàng có dữ liệu > số hàng không có dữ liệu
    3. Lựa chọn các cột useful (chỉ giữ lại cột numeric)
    4. Tìm tên cột thực sự
    5. Đưa ra mô tả cho từng cột kết hợp tên bảng, thời kỳ và đơn vị tính
    6. Xác định động label_column và value_column
    7. In Output JSON ra console phục vụ kiểm thử.
    """
    cfg = cfg or default_config
    start_time = time.time()

    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    top_candidates = state.get("top_k_candidates", discovered_tables[:5])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    print(f"\n" + "=" * 65)
    print(f"🔍 [Node 3: SCHEMA MAPPER] Bắt đầu phân tích Schema theo dữ liệu thực tế...")
    print(f"=" * 65)
    print(f"   - Tiêu chí phụ cần tìm: '{tieu_chi_phu or '(không có)'}'")

    column_mapping: Dict[str, Any] = {}
    schema: Dict[str, Any] = {"useful_columns": [], "sub_sections": []}

    if not discovered_tables:
        print(f"   ⚠️ [Schema Mapper] Không có bảng dữ liệu đầu vào để ánh xạ.")
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["schema_mapper"] = round(latency, 3)
        return {
            **state,
            "column_mapping": {},
            "schema": schema,
            "top_k_candidates": [],
            "status": "pending",
            "node_latencies": node_latencies,
        }

    first_table = discovered_tables[0]
    csv_path = first_table.get("csv_path", "")
    table_name = first_table.get("Ten_Bang", Path(csv_path).stem if csv_path else "unknown")
    metadata_set = set(METADATA_HEADER_COLUMNS)

    try:
        df_full = pd.read_csv(csv_path)
        raw_columns = list(df_full.columns)
        print(f"   📋 Đọc bảng '{table_name}' từ: {csv_path}")

        # Trích xuất đơn vị tính từ metadata nếu có
        don_vi_tinh = None
        if "Don_Vi_Tinh" in df_full.columns and len(df_full) > 0:
            first_dvt = df_full.iloc[0]["Don_Vi_Tinh"]
            if not _is_cell_empty(first_dvt):
                don_vi_tinh = str(first_dvt).strip()

        # ── 1, 2, 3, 4. Trích xuất useful_columns qua kiểm tra dữ liệu từng cột ──
        all_useful_detected = _extract_useful_columns(df_full, metadata_cols=metadata_set)

        # ── 5. Xác định động label_column và value_column (Không dùng danh sách ưu tiên) ──
        label_col = _find_label_column(all_useful_detected, raw_columns)

        # Chỉ giữ lại các cột numeric trong useful_columns (loại bỏ các cột auxiliary code và text)
        useful_columns = [
            c for c in all_useful_detected 
            if c.get("data_type") == "numeric" 
            and not c.get("is_aux_code", False)
            and not AUXILIARY_COL_REGEX.match(str(c.get("column_name", "")).strip())
            and not AUXILIARY_COL_REGEX.match(str(c.get("raw_column", "")).strip())
            and str(c.get("column_name", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
            and str(c.get("raw_column", "")).strip().lower() not in _AUXILIARY_CODE_COLUMNS
        ]
        if not useful_columns:
            useful_columns = [c for c in all_useful_detected if c.get("raw_column") != label_col]

        value_col = _find_value_column(useful_columns, label_col, tieu_chi_phu, raw_columns)

        label_col_idx = raw_columns.index(label_col) if (label_col and label_col in raw_columns) else 0
        value_col_idx = raw_columns.index(value_col) if (value_col and value_col in raw_columns) else 1

        column_mapping = {
            "label_column": label_col,
            "value_column": value_col,
            "label_column_idx": label_col_idx,
            "value_column_idx": value_col_idx,
            "all_columns": str(raw_columns),
            "all_useful_columns": [c["raw_column"] for c in useful_columns],
        }

        print(f"\n   🎯 [Workflow Step 6: Xác định Cột Nhãn & Cột Giá Trị]")
        print(f"      • Cột nhãn (label_column): '{label_col}'")
        print(f"      • Cột giá trị (value_column): '{value_col}'")

        # ── 6. Phân tích Sub-sections ──
        sub_sections = _extract_sub_sections(df_full, label_col, metadata_set)
        if sub_sections:
            print(f"   📂 Phát hiện {len(sub_sections)} danh mục con (Sub-sections).")

        # ── 7. Sinh mô tả ngữ nghĩa cho các cột bằng LLM ──
        useful_columns = _enrich_column_descriptions(cfg, useful_columns, table_name, don_vi_tinh=don_vi_tinh)

        schema = {
            "useful_columns": useful_columns,
            "sub_sections": sub_sections,
        }

        # Ánh xạ schema cho toàn bộ danh sách candidate tables qua map_multi_tables
        top_candidates = map_multi_tables(top_candidates, parsed_query, cfg=cfg)

        # ── 8. In Output JSON phục vụ kiểm thử ──
        output_json = {
            "column_mapping": column_mapping,
            "schema": schema,
        }

        print(f"\n" + "-" * 65)
        print(f"📄 [SCHEMA MAPPER OUTPUT JSON - DÙNG CHO KIỂM THỬ]:")
        print(json.dumps(output_json, indent=2, ensure_ascii=False))
        print("-" * 65)

    except Exception as e:
        print(f"   ⚠️ [Schema Mapper] Lỗi trong quá trình phân tích schema: {e}")
        import traceback
        traceback.print_exc()

    latency = time.time() - start_time
    node_latencies = state.get("node_latencies", {})
    node_latencies["schema_mapper"] = round(latency, 3)

    print(f"\n   ⏱️ [Schema Mapper] Hoàn thành trong {latency:.3f}s")
    print(f"=" * 65 + "\n")

    return {
        **state,
        "column_mapping": column_mapping,
        "schema": schema,
        "top_k_candidates": top_candidates,
        "status": "pending",
        "node_latencies": node_latencies,
    }



✅ Node 3: Schema Mapper Node loaded!


### Node 4: Code Generator & Reflection Node
Sinh mã Pandas xử lý câu hỏi tài chính và hỗ trợ Reflection Debugging Loop khi xảy ra lỗi.

In [28]:
"""Node 4: Code Generation & Reflection Node.
Sinh code Python/Pandas dựa trên mục tiêu (trich_xuat/tinh_tong/so_sanh),
cột mapping, và bảng dữ liệu đã tìm được.
Tất cả các prompt, quy tắc và template được nạp từ file prompts/code_generator.yaml & prompts/reflection.yaml.
"""

import re
import time
import yaml
import pandas as pd
from pathlib import Path
from typing import Dict, Any, Optional, List
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config
from pipeline.src.llm_provider import get_llm
from pipeline.src.nodes.data_discovery import _extract_table_schema


def load_yaml_prompt(cfg: Config, filename: str) -> Dict[str, Any]:
    """Load prompt template YAML with fallback to Kaggle global prompt dicts."""
    try:
        if hasattr(cfg, "get_prompt_path"):
            prompt_path = cfg.get_prompt_path(filename)
            if prompt_path and Path(prompt_path).exists():
                with open(prompt_path, "r", encoding="utf-8") as f:
                    return yaml.safe_load(f)
    except Exception:
        pass

    if "code_generator" in filename and "PROMPT_CODE_GENERATOR" in globals():
        return globals()["PROMPT_CODE_GENERATOR"]
    elif "reflection" in filename and "PROMPT_REFLECTION" in globals():
        return globals()["PROMPT_REFLECTION"]

    raise FileNotFoundError(f"Prompt file {filename} not found and no global fallback available.")


def clean_python_code(raw_code: str) -> str:
    """Extract clean Python code from LLM response, stripping markdown and conversational text."""
    if not raw_code:
        return ""

    import ast

    # Step 1: Check markdown python blocks
    pattern = r"```(?:python)?\s*\n?(.*?)\n?```"
    matches = re.findall(pattern, raw_code, re.DOTALL)
    for match in matches:
        candidate = match.strip()
        try:
            ast.parse(candidate)
            return candidate
        except SyntaxError:
            pass

    # Step 2: If no valid code block found, check inside <think> tags
    think_match = re.search(r"<think>(.*?)</think>", raw_code, re.DOTALL)
    if think_match:
        think_text = think_match.group(1).strip()
        think_matches = re.findall(pattern, think_text, re.DOTALL)
        for tm in think_matches:
            candidate = tm.strip()
            try:
                ast.parse(candidate)
                return candidate
            except SyntaxError:
                pass
        lines = think_text.splitlines()
        for idx, line in enumerate(lines):
            l = line.strip()
            if l.startswith("import ") or l.startswith("file_path =") or l.startswith("df ="):
                candidate = "\n".join(lines[idx:]).strip()
                try:
                    ast.parse(candidate)
                    return candidate
                except SyntaxError:
                    pass

    # Step 3: Strip conversational text before code in raw_code
    cleaned = raw_code.strip()
    if cleaned.startswith("```") and cleaned.endswith("```"):
        cleaned = cleaned[3:-3].strip()

    lines = cleaned.splitlines()
    for idx, line in enumerate(lines):
        l = line.strip()
        if l.startswith("import ") or l.startswith("def ") or l.startswith("file_path") or l.startswith("df =") or l.startswith("result ="):
            candidate = "\n".join(lines[idx:]).strip()
            try:
                ast.parse(candidate)
                return candidate
            except SyntaxError:
                pass

    return ""


# Các cột metadata/thông tin chung ở đầu file CSV
_METADATA_COLUMNS = {
    "Ma_Doanh_Nghiep", "Ten_Doanh_Nghiep", "Nam_Tai_Chinh",
    "Loai_Bao_Cao", "Ten_Bang", "Don_Vi_Tinh", "Tep_Nguon"
}

_PERSON_BLOCKLIST = {
    "và các khoản khác", "các khoản khác", "đã phát hành", "đã phát hành của",
    "quản lý doanh nghiệp", "phát hành thêm", "công ty mẹ", "chi phí",
    "doanh thu", "lợi nhuận", "vốn chủ sở hữu", "vốn cổ phần", "tổng giám đốc",
    "hội đồng quản trị", "chứng khoán fpt", "báo cáo tài chính", "tổng công ty",
    "hàng không vietjet", "ngân hàng tmcp", "tập đoàn", "công ty cổ phần"
}


def extract_person_name(user_query: str) -> Optional[str]:
    """Extract executive person name from financial query, rejecting financial line item false positives."""
    if not user_query:
        return None
    q_lower = user_query.lower()
    if any(k in q_lower for k in [
        "chi phí lương", "quỹ lương", "vốn cổ phần", "cổ phần đã phát hành",
        "tỷ lệ sở hữu", "quyền biểu quyết", "tỷ lệ biểu quyết", "chi phí quản lý"
    ]):
        return None

    # Priority 1: Match title prefix + Capitalized Name
    m = re.search(
        r"(?i:(?:thành viên\s+(?:hđqt|hội đồng quản trị|bqt|bks|ban kiểm soát|ban tổng giám đốc|ban giám đốc)|chủ tịch(?:\s+hđqt)?|tổng giám đốc|tgđ|phó tổng giám đốc|phó tgđ|ông|bà))\s+([A-ZÀ-Ỹ][a-zà-ỹ]+(?:\s+[A-ZÀ-Ỹ][a-zà-ỹ]+){1,3})",
        user_query
    )
    if m:
        cand = m.group(1).strip()
        if cand.lower() not in _PERSON_BLOCKLIST:
            return cand

    # Priority 2: Remuneration / Salary of person
    m2 = re.search(
        r"(?i:(?:thù lao|tiền lương|thưởng|thu nhập)\s+(?:của\s+)?(?:ông\s+|bà\s+)?)([A-ZÀ-Ỹ][a-zà-ỹ]+(?:\s+[A-ZÀ-Ỹ][a-zà-ỹ]+){1,3})",
        user_query
    )
    if m2:
        cand = m2.group(1).strip()
        if cand.lower() not in _PERSON_BLOCKLIST and not any(cand.lower().startswith(b) for b in ["ctcp", "ngân hàng", "công ty", "tập đoàn", "chứng khoán"]):
            return cand
    return None


def generate_fallback_code(
    muc_tieu: str,
    noi_dung: str,
    label_col: str,
    value_col: str,
    so_nam: List[str],
    discovered_tables: List[Dict[str, Any]],
    person_name: Optional[str] = None,
    tieu_chi_phu: Optional[str] = None,
    user_query: str = "",
    label_col_idx: Optional[int] = None,
    value_col_idx: Optional[int] = None,
) -> str:
    """Generate concise Pandas extraction code using column position indices from Schema Mapper."""
    escaped_noi_dung = (noi_dung or "").replace("'", "\\'").strip()
    escaped_person = (person_name or "").replace("'", "\\'").strip()
    escaped_tieu_chi = (tieu_chi_phu or "").replace("'", "\\'").strip()

    is_growth = any(k in user_query.lower() for k in ["tăng trưởng", "tốc độ", "%", "thay đổi"])

    lbl_ref = f"df.iloc[:, {label_col_idx}]" if label_col_idx is not None else f"df['{label_col}']"
    val_arg = value_col_idx if value_col_idx is not None else f"'{value_col}'"

    code_lines = [
        "import pandas as pd",
        "import numpy as np",
        "df = pd.read_csv(file_path)",
        f"# Truy vấn trực tiếp theo vị trí cột: label_col='{label_col}', value_col='{value_col}'",
    ]

    if person_name:
        code_lines.extend([
            f"# Lọc trực tiếp theo thực thể nhân sự: '{escaped_person}'",
            f"filtered_df = df[{lbl_ref}.astype(str).str.contains('{escaped_person}', case=False, na=False, regex=False)]",
            "if not filtered_df.empty:",
            "    match_row = filtered_df.iloc[0]",
            f"    result = extract_value(match_row, {val_arg}, _df=df, _row_idx=match_row.name)",
            "else:",
            f"    raise ValueError(\"Person '{escaped_person}' not found in table\")",
        ])
        return "\n".join(code_lines)

    if muc_tieu == "so_sanh" and len(so_nam) >= 2:
        y_sorted = sorted(so_nam, key=lambda y: int(y) if str(y).isdigit() else 0)
        y_old, y_new = y_sorted[0], y_sorted[-1]

        code_lines.extend([
            f"# So sánh giữa các năm {so_nam}",
            f"search_key = '{escaped_noi_dung}'",
            f"filtered_df = df[{lbl_ref}.astype(str).str.contains(search_key, case=False, na=False, regex=False)]",
            "if filtered_df.empty:",
            f"    tokens = [t for t in search_key.split() if len(t) > 2 and t.lower() not in ['tổng', 'chi_phí', 'doanh_thu', 'năm', 'của', 'và', 'các', 'khoản', 'theo']]",
            "    for t in tokens:",
            f"        filtered_df = df[{lbl_ref}.astype(str).str.contains(t, case=False, na=False, regex=False)]",
            "        if not filtered_df.empty:",
            "            break",
            "if not filtered_df.empty:",
            "    match_row = filtered_df.iloc[0]",
            "    _meta = {'Ma_Doanh_Nghiep', 'Ten_Doanh_Nghiep', 'Nam_Tai_Chinh', 'Loai_Bao_Cao', 'Ten_Bang', 'Don_Vi_Tinh', 'Tep_Nguon'}",
            "    cols = [c for c in df.columns if c not in _meta]",
            f"    col_new = next((c for c in cols if '{y_new}' in str(c)), {val_arg})",
            f"    col_old = next((c for c in cols if '{y_old}' in str(c)), cols[1] if len(cols) > 1 else col_new)",
            "    val_new = extract_value(match_row, col_new, _df=df, _row_idx=match_row.name)",
            "    val_old = extract_value(match_row, col_old, _df=df, _row_idx=match_row.name)",
        ])
        if is_growth:
            code_lines.append("    result = ((val_new - val_old) / abs(val_old)) * 100 if val_old != 0 else 0.0")
        else:
            code_lines.append("    result = val_new - val_old")
        code_lines.extend([
            "else:",
            f"    raise ValueError(\"Metric '{escaped_noi_dung}' not found in table\")",
        ])
        return "\n".join(code_lines)

    # Standard direct extraction
    code_lines.extend([
        f"search_key = '{escaped_noi_dung}'",
        f"filtered_df = df[{lbl_ref}.astype(str).str.contains(search_key, case=False, na=False, regex=False)]",
        "if filtered_df.empty:",
        "    tokens = [t for t in search_key.split() if len(t) > 2 and t.lower() not in ['tổng', 'chi_phí', 'doanh_thu', 'năm', 'của', 'và', 'các', 'khoản', 'theo', 'công', 'mẹ', 'đã', 'phát', 'hành']]",
        "    for t in tokens:",
        f"        filtered_df = df[{lbl_ref}.astype(str).str.contains(t, case=False, na=False, regex=False)]",
        "        if not filtered_df.empty:",
        "            break",
        "if not filtered_df.empty:",
        "    match_row = filtered_df.iloc[0]",
        f"    result = extract_value(match_row, {val_arg}, _df=df, _row_idx=match_row.name)",
        "else:",
        f"    raise ValueError(\"Metric '{escaped_noi_dung}' not found in table\")",
    ])
    return "\n".join(code_lines)


def _resolve_label_column(
    table_schema: List[str],
    first_row_values: Optional[Dict[str, str]] = None,
    column_mapping: Optional[Dict[str, str]] = None,
    schema: Optional[Dict[str, Any]] = None,
) -> str:
    """Chọn cột nhãn (chứa tên chỉ tiêu tài chính) dựa trên column_mapping và schema thực tế.

    Logic:
    1. Nếu column_mapping có label_column và cột đó thực sự tồn tại trong table_schema -> dùng nó.
    2. Nếu có schema useful_columns -> lấy cột dạng text đầu tiên.
    3. Lấy cột đầu tiên trong table_schema không thuộc _METADATA_COLUMNS.
    4. Fallback: column_mapping['label_column'] nếu có, hoặc cột đầu tiên.
    """
    column_mapping = column_mapping or {}
    schema = schema or {}

    # 1. Kiểm tra column_mapping nếu cột đó thực sự có trong table_schema
    mapped_label = column_mapping.get("label_column")
    if mapped_label and mapped_label in table_schema:
        return mapped_label

    # 2. Kiểm tra useful_columns trong schema
    useful_cols = schema.get("useful_columns", [])
    text_cols = [c.get("raw_column") for c in useful_cols if c.get("data_type") == "text"]
    for tc in text_cols:
        if tc in table_schema:
            return tc

    if not table_schema:
        return mapped_label or "0"

    # 3. Lấy cột đầu tiên không phải metadata
    non_meta_cols = [c for c in table_schema if c not in _METADATA_COLUMNS]
    if non_meta_cols:
        return non_meta_cols[0]

    return mapped_label or table_schema[0]


def _resolve_value_column(
    table_schema: List[str],
    first_row_values: Dict[str, str],
    parsed_query: Dict[str, Any],
    column_mapping: Dict[str, str],
    label_col: Optional[str] = None,
    schema: Optional[Dict[str, Any]] = None,
) -> str:
    """Tự chọn cột giá trị dựa trên schema thực tế, tiêu chí phụ và dữ liệu mẫu."""
    fallback = column_mapping.get("value_column", "Năm nay")
    schema = schema or {}

    if not table_schema:
        return fallback

    label_col = label_col or column_mapping.get("label_column", "")

    # Lọc cột dữ liệu (loại bỏ metadata + label)
    data_cols = [
        c for c in table_schema
        if c not in _METADATA_COLUMNS and c != label_col
    ]

    if not data_cols:
        return fallback

    if len(data_cols) == 1:
        return data_cols[0]

    tieu_chi_phu = parsed_query.get("tieu_chi_phu", "")
    noi_dung = parsed_query.get("noi_dung", "")

    # 1. So sánh tiêu chí phụ trực tiếp với tên các cột dữ liệu
    if tieu_chi_phu:
        tcp_lower = str(tieu_chi_phu).strip().lower()
        for col in data_cols:
            col_str = str(col).strip().lower()
            if tcp_lower in col_str or col_str in tcp_lower:
                print(f"   🎯 [Code Generator] Tự chọn cột '{col}' khớp với tiêu chí phụ '{tieu_chi_phu}'")
                return col

        m_year = re.search(r"\b(19|20)\d{2}\b", tcp_lower)
        if m_year:
            target_year = m_year.group(0)
            for col in data_cols:
                if target_year in str(col):
                    print(f"   🎯 [Code Generator] Tự chọn cột '{col}' khớp với năm '{target_year}' từ tiêu chí phụ")
                    return col

        if any(kw in tcp_lower for kw in ["%", "phần trăm", "tỷ lệ", "biểu quyết", "sở hữu"]):
            for col in data_cols:
                if any(k in str(col).lower() for k in ["%", "tỷ lệ", "biểu quyết", "sở hữu"]):
                    print(f"   🎯 [Code Generator] Tự chọn cột '{col}' khớp với chỉ tiêu tỷ lệ % từ tiêu chí phụ")
                    return col

    # 2. So sánh với useful_columns từ schema
    useful_columns = schema.get("useful_columns", [])
    if useful_columns:
        for uc in useful_columns:
            c_name = str(uc.get("column_name", ""))
            r_name = str(uc.get("raw_column", ""))
            if tieu_chi_phu and (str(tieu_chi_phu).lower() in c_name.lower() or str(tieu_chi_phu).lower() in r_name.lower()):
                target_col = r_name if r_name in data_cols else c_name
                if target_col in data_cols:
                    print(f"   🎯 [Code Generator] Tự chọn cột '{target_col}' từ useful_columns dựa trên '{tieu_chi_phu}'")
                    return target_col

    # 3. Nếu cột có tên là số (0, 1, 2...), dùng first_row_values để đoán cột đúng
    numeric_named_cols = [c for c in data_cols if str(c).strip().isdigit()]
    if numeric_named_cols and first_row_values and tieu_chi_phu:
        tcp_lower = str(tieu_chi_phu).strip().lower()
        for col in data_cols:
            val = str(first_row_values.get(str(col), "")).lower()
            if tcp_lower in val:
                print(f"   🎯 [Code Generator] Tự chọn cột '{col}' (hàng 1: '{val}') từ tiêu chí phụ '{tieu_chi_phu}'")
                return col

    if fallback in data_cols:
        return fallback

    return data_cols[0] if data_cols else fallback


def _build_files_context(
    discovered_tables: List[Dict[str, Any]],
    column_mapping: Dict[str, str],
    table_schema: List[str] = None,
    first_row_values: Dict[str, str] = None,
    schema: Dict[str, Any] = None,
) -> str:
    """Build context string describing available files for code generation."""
    if not discovered_tables:
        return "Không có bảng dữ liệu."

    lines = []
    for i, tbl in enumerate(discovered_tables):
        csv_path = tbl.get("csv_path", "")
        ten_bang = tbl.get("Ten_Bang", "N/A")
        nam = tbl.get("Nam_Tai_Chinh", "N/A")
        escaped_path = csv_path.replace('\\', '\\\\')
        lines.append(
            f"- File {i+1} (Năm {nam}):\n"
            f"  Đường dẫn: '{escaped_path}'\n"
            f"  Tên bảng: {ten_bang}\n"
        )

    lines.append(f"\nColumn Mapping: {column_mapping}")

    # Thêm thông tin schema bảng
    if table_schema:
        lines.append(f"\nSchema bảng (tên các cột): {table_schema}")

    # Thêm giá trị hàng đầu tiên nếu có cột tên là số
    if first_row_values:
        lines.append(f"\nGiá trị hàng đầu tiên (giúp hiểu ý nghĩa cột số):")
        for col, val in first_row_values.items():
            lines.append(f"  Cột '{col}' → '{val}'")

    # Thêm schema analysis context (useful_columns + sub_sections)
    schema = schema or {}
    useful_columns = schema.get("useful_columns", [])
    sub_sections = schema.get("sub_sections", [])

    if useful_columns:
        lines.append(f"\nSCHEMA PHÂN TÍCH BẢNG - CỘT GIÁ TRỊ HỮU DỤNG:")
        for uc in useful_columns:
            col_name = uc.get("column_name", "")
            col_desc = uc.get("column_description", "")
            desc_str = f" — {col_desc}" if col_desc else ""
            lines.append(f"  • '{col_name}'{desc_str}")

    if sub_sections:
        lines.append(f"\nSCHEMA PHÂN TÍCH BẢNG - DANH MỤC CON (SUB-SECTIONS):")
        for sec in sub_sections:
            sec_name = sec.get("section_name", "")
            sec_range = sec.get("range", [])
            total_val = sec.get("total_value")
            total_str = f", total_value={total_val}" if total_val is not None else ", total_value=N/A"
            lines.append(f"  • '{sec_name}' (hàng {sec_range[0]}–{sec_range[1]}{total_str})")

    # Add sample row labels from the label column so LLM sees actual text entries
    label_col = column_mapping.get("label_column")
    if discovered_tables and label_col:
        from pathlib import Path
        c_path = discovered_tables[0].get("csv_path")
        if c_path and Path(c_path).exists():
            try:
                sub_df = pd.read_csv(c_path)
                if label_col in sub_df.columns:
                    sample_labels = sub_df[label_col].dropna().astype(str).head(15).tolist()
                    if sample_labels:
                        lines.append(f"\nMẪU NHÃN HÀNG THỰC TẾ TRONG CỘT '{label_col}':")
                        for lbl in sample_labels:
                            lines.append(f"  • '{lbl}'")
            except Exception:
                pass

    return "\n".join(lines)


def code_generator_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 4: Sinh code Pandas hoặc sửa code lỗi (Reflection Loop).

    Nạp prompt và hướng dẫn từ YAML, hỗ trợ các mục tiêu (trich_xuat, tinh_tong, so_sanh).

    Args:
        state: Current AgentState
        cfg: Config instance

    Returns:
        Updated AgentState with 'generated_code'
    """
    cfg = cfg or default_config
    start_time = time.time()

    user_query = state.get("user_query", "")
    parsed_query = state.get("parsed_query", {})
    discovered_tables = state.get("discovered_tables", [])
    column_mapping = state.get("column_mapping", {})
    table_schema = state.get("table_schema", [])
    first_row_values = state.get("first_row_values", {})
    schema = state.get("schema", {})
    if (not column_mapping or not schema) and discovered_tables:
        from pipeline.src.nodes.schema_mapper import schema_mapper_node
        try:
            temp_state = schema_mapper_node(state, cfg)
            column_mapping = temp_state.get("column_mapping", {})
            schema = temp_state.get("schema", {})
            state["column_mapping"] = column_mapping
            state["schema"] = schema
        except Exception as e:
            print(f"⚠️ Inline schema mapping failed: {e}")
    error_traceback = state.get("error_traceback")
    retry_count = state.get("retry_count", 0)

    # Extract parsed query fields
    muc_tieu = parsed_query.get("muc_tieu", "trich_xuat")
    noi_dung = parsed_query.get("noi_dung", "")
    ten_cong_ty = parsed_query.get("ten_cong_ty", "")
    so_nam = parsed_query.get("so_nam", [])
    tieu_chi_phu = parsed_query.get("tieu_chi_phu")

    # Detect person name if query involves remuneration / salary / executive personnel (Q15)
    person_name = parsed_query.get("ten_nhan_su") or parsed_query.get("person_name")
    if not person_name and isinstance(user_query, str):
        person_name = extract_person_name(user_query)

    # Get column names from mapping, sử dụng schema thực tế nếu có
    label_col = _resolve_label_column(table_schema, first_row_values, column_mapping, schema=schema)
    value_col = _resolve_value_column(table_schema, first_row_values, parsed_query, column_mapping, label_col, schema=schema)

    label_col_idx = column_mapping.get("label_column_idx")
    if label_col_idx is None and label_col in table_schema:
        label_col_idx = table_schema.index(label_col)

    value_col_idx = column_mapping.get("value_column_idx")
    if value_col_idx is None and value_col in table_schema:
        value_col_idx = table_schema.index(value_col)

    print(f"   📋 [Code Generator] Schema: {table_schema}")
    print(f"   📋 [Code Generator] Label col: '{label_col}' (idx={label_col_idx}), Value col: '{value_col}' (idx={value_col_idx})")
    if person_name:
        print(f"   👤 [Code Generator] Nhận diện nhân sự/lãnh đạo: '{person_name}'")
    if first_row_values:
        print(f"   📋 [Code Generator] First row values: {first_row_values}")

    # Build files context (bao gồm schema và first_row info)
    files_context = _build_files_context(discovered_tables, column_mapping, table_schema, first_row_values, schema=schema)
    if person_name:
        files_context += f"\n\n👤 THỰC THỂ NHÂN SỰ/LÃNH ĐẠO: '{person_name}'. ƯU TIÊN LỌC THEO TÊN NÀY TRÊN CỘT NHÃN '{label_col}'."

    core_tokens = [t for t in re.findall(r"\w+", noi_dung) if len(t) > 2 and t.lower() not in ["tổng", "tổng_số", "số_dư", "chi_phí", "giá_trị", "chỉ_tiêu", "năm", "báo", "cáo"]]
    if core_tokens:
        files_context += f"\n🔑 TỪ KHÓA CỐT LÕI (DÙNG CHO MULTI-STAGE QUERY NẾU CẤP 1 RỖNG): {core_tokens}"

    # Build file path variables for code
    paths_str = ""
    if discovered_tables:
        top_csv = discovered_tables[0]["csv_path"].replace('\\', '\\\\')
        paths_str = f"file_path = '{top_csv}'\n"

        year_paths = {}
        for tbl in discovered_tables:
            nam = str(tbl.get("Nam_Tai_Chinh", "")).strip()
            csv_p = tbl.get("csv_path", "").replace('\\', '\\\\')
            if nam and nam not in year_paths:
                year_paths[nam] = csv_p
        if len(year_paths) > 1:
            for nam, csv_p in year_paths.items():
                paths_str += f"file_path_{nam} = '{csv_p}'\n"
    paths_str = paths_str.strip()

    try:
        # Scenario A: Initial Code Generation
        if not error_traceback or retry_count == 0:
            prompt_data = load_yaml_prompt(cfg, "code_generator.yaml")
            system_prompt = prompt_data["system_prompt"]
            few_shots = prompt_data.get("few_shot_examples", [])
            goal_descs = prompt_data.get("goal_descriptions", {})
            goal_instructions = prompt_data.get("goal_instructions", {})

            messages = [SystemMessage(content=system_prompt)]
            for ex in few_shots:
                messages.append(
                    HumanMessage(
                        content=f"Yêu cầu: {ex['user_query']}\n"
                                f"File Path: {ex['file_path']}\n"
                                f"Column Mapping: {ex['column_mapping']}"
                    )
                )
                messages.append(AIMessage(content=ex["generated_code"]))

            # Retrieve goal description and goal instruction from YAML
            goal_desc = goal_descs.get(muc_tieu, muc_tieu)
            goal_inst_template = goal_instructions.get(muc_tieu, "")
            goal_inst = goal_inst_template.format(
                noi_dung=noi_dung, label_col=label_col, value_col=value_col
            ) if goal_inst_template else ""

            # Format user prompt from YAML template
            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu_desc=goal_desc,
                noi_dung=noi_dung,
                ten_cong_ty=ten_cong_ty,
                so_nam=so_nam,
                tieu_chi_phu=tieu_chi_phu or "(không có)",
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                goal_instruction=goal_inst,
            )

            messages.append(HumanMessage(content=human_content))

        # Scenario B: Reflection Debugging Loop (retry_count > 0)
        else:
            prompt_data = load_yaml_prompt(cfg, "reflection.yaml")
            system_prompt = prompt_data["system_prompt"]

            if len(discovered_tables) > 1:
                # Multi-table fallback: rotate candidate tables on retry
                shift = retry_count % len(discovered_tables)
                discovered_tables = discovered_tables[shift:] + discovered_tables[:shift]
                first_table_path = discovered_tables[0]["csv_path"]
                schema_info = _extract_table_schema(first_table_path)
                table_schema = schema_info["table_schema"]
                first_row_values = schema_info["first_row_values"]
                state["discovered_tables"] = discovered_tables
                state["matched_table_path"] = first_table_path
                state["table_schema"] = table_schema
                state["first_row_values"] = first_row_values
                from pipeline.src.nodes.schema_mapper import schema_mapper_node
                try:
                    temp_state = schema_mapper_node(state, cfg)
                    column_mapping = temp_state.get("column_mapping", {})
                    schema = temp_state.get("schema", {})
                    state["column_mapping"] = column_mapping
                    state["schema"] = schema
                    label_col = _resolve_label_column(table_schema, first_row_values, column_mapping, schema=schema)
                    value_col = _resolve_value_column(table_schema, first_row_values, parsed_query, column_mapping, label_col, schema=schema)
                    files_context = _build_files_context(discovered_tables, column_mapping, table_schema, first_row_values, schema=schema)
                except Exception as e:
                    print(f"⚠️ Multi-table fallback schema mapping failed: {e}")

            print(f"🔄 [Reflection Loop] Đang sửa lỗi mã nguồn (Lần {retry_count})...")
            print(f"   - Bảng được chọn: {discovered_tables[0].get('Ten_Bang')} ({discovered_tables[0].get('csv_path')})")
            print(f"   - Traceback Lỗi:\n{error_traceback.strip()}")

            retry_forcing_msg = (
                f"Execution failed with error: {error_traceback.strip()}\n"
                "CRITICAL: The string you used in `str.contains()` was NOT found in the table. "
                "DO NOT output the exact same code again! "
                "You MUST change your search strategy: shorten the search string in `str.contains(..., regex=False)` to a single core keyword from the metric, or inspect the sample row labels below."
            )

            sample_labels = []
            if discovered_tables:
                from pathlib import Path
                import pandas as pd
                for tbl in discovered_tables:
                    c_path = tbl.get("csv_path")
                    if c_path and Path(c_path).exists():
                        try:
                            sub_df = pd.read_csv(c_path)
                            if label_col in sub_df.columns:
                                labels = sub_df[label_col].dropna().astype(str).head(20).tolist()
                                sample_labels.append(f"Mẫu chỉ tiêu thực tế trong file '{Path(c_path).name}':\n{labels}")
                        except Exception:
                            pass
            sample_labels_str = "\n\n".join(sample_labels) if sample_labels else ""

            user_template = prompt_data.get("user_prompt_template", "")
            human_content = user_template.format(
                user_query=user_query,
                muc_tieu=muc_tieu,
                noi_dung=noi_dung,
                files_context=files_context,
                label_col=label_col,
                value_col=value_col,
                paths_str=paths_str,
                sample_labels_str=sample_labels_str,
                previous_code=state.get('generated_code', ''),
                error_traceback=f"{error_traceback.strip()}\n\n{retry_forcing_msg}",
            )

            messages = [
                SystemMessage(content=system_prompt),
                HumanMessage(content=human_content)
            ]

        # Call LLM
        llm = get_llm(cfg=cfg, temperature=0.0)
        response = llm.invoke(messages)
        raw_text = response.content if isinstance(response.content, str) else str(response.content)

        # Extract and print thoughts
        think_match = re.search(r"<think>(.*?)</think>", raw_text, re.DOTALL)
        if think_match:
            thought = think_match.group(1).strip()
            indented_thought = thought.replace('\n', '\n  ')
            print(f"💭 [Tư duy - Code Generator]:\n  {indented_thought}")
        else:
            code_start = raw_text.find("```")
            if code_start > 10:
                thought = raw_text[:code_start].strip()
                indented_thought = thought.replace('\n', '\n  ')
                print(f"💭 [Tư duy - Code Generator]:\n  {indented_thought}")

        code = clean_python_code(raw_text)

        if code:
            try:
                ast.parse(code)
            except Exception:
                code = ""

        if not code:
            print("⚠️ [Code Generator] LLM không sinh code hợp lệ -> Kích hoạt Rule-based Fallback Generator...")
            code = generate_fallback_code(
                muc_tieu=muc_tieu,
                noi_dung=noi_dung,
                label_col=label_col,
                value_col=value_col,
                so_nam=so_nam,
                discovered_tables=discovered_tables,
                person_name=person_name,
                tieu_chi_phu=tieu_chi_phu,
                user_query=user_query,
                label_col_idx=label_col_idx,
                value_col_idx=value_col_idx,
            )

        print(f"📊 [Kết quả - Code Generator] Mã Python sinh ra:\n```python\n{code}\n```\n")

        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)

        return {
            **state,
            "generated_code": code,
            "status": "pending",
            "node_latencies": node_latencies,
        }

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["code_generator"] = round(latency, 3)

        print(f"⚠️ [Code Generator] LLM không phản hồi ({e}). Đang sử dụng Rule-based Fallback Generator...")
        fallback_code = generate_fallback_code(
            muc_tieu=muc_tieu,
            noi_dung=noi_dung,
            label_col=label_col,
            value_col=value_col,
            so_nam=so_nam,
            discovered_tables=discovered_tables,
            person_name=person_name,
            tieu_chi_phu=tieu_chi_phu,
            user_query=user_query,
            label_col_idx=label_col_idx,
            value_col_idx=value_col_idx,
        )
        print(f"📊 [Kết quả Fallback - Code Generator] Mã Python sinh ra:\n```python\n{fallback_code}\n```\n")

        return {
            **state,
            "generated_code": fallback_code,
            "status": "pending",
            "node_latencies": node_latencies,
        }

### Node 5: AST Sandbox & Executor Node
Thực thi mã Python trong môi trường Sandbox AST an toàn và thu thập kết quả `result`.

In [29]:
"""Node 5: AST Sandbox & Execution Node.
Executes generated Pandas code safely inside a restricted AST sandbox environment.
"""

import ast
import re
import sys
import time
import traceback
from pathlib import Path
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional, List

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config


class SecurityError(Exception):
    """Raised when generated code contains forbidden AST nodes."""
    pass


FORBIDDEN_AST_NODES = (
    ast.Import,
    ast.ImportFrom,
)

FORBIDDEN_BUILTINS = {
    "eval", "exec", "__import__", "open", "compile",
    "globals", "locals", "input", "breakpoint"
}

ALLOWED_MODULES = {"pandas", "pd", "numpy", "np", "datetime", "math", "re", "pathlib", "Path"}


def validate_ast(code_str: str) -> None:
    """Validate Python code against AST safety rules."""
    tree = ast.parse(code_str)

    for node in ast.walk(tree):
        # Check forbidden imports unless in ALLOWED_MODULES
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_MODULES:
                    raise SecurityError(f"Importing forbidden module: '{alias.name}'")

        elif isinstance(node, ast.ImportFrom):
            if node.module and node.module.split(".")[0] not in ALLOWED_MODULES:
                raise SecurityError(f"Importing from forbidden module: '{node.module}'")

        # Check forbidden built-in function calls
        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name) and node.func.id in FORBIDDEN_BUILTINS:
                raise SecurityError(f"Call to forbidden function: '{node.func.id}'")


def format_result(result: Any) -> Any:
    """Format DataFrame, Series, or scalar result for JSON serialization."""
    if isinstance(result, pd.DataFrame):
        # Limit rows for output size
        df_sub = result.head(100)
        return {
            "type": "dataframe",
            "shape": list(result.shape),
            "columns": list(result.columns),
            "data": df_sub.to_dict(orient="records"),
        }
    elif isinstance(result, pd.Series):
        s_sub = result.head(100)
        return {
            "type": "series",
            "name": str(result.name) if result.name else "result",
            "data": s_sub.to_dict(),
        }
    elif isinstance(result, (int, float, str, bool, list, dict)):
        return {
            "type": "scalar",
            "data": result,
        }
    else:
        return {
            "type": "other",
            "data": str(result),
        }


def sanitize_code_str(code_str: str) -> str:
    """Pre-process and fix common LLM-generated code syntax bugs before execution."""
    if not code_str:
        return ""

    # Fix indentation: remove common leading indentation if entire block is indented
    lines = code_str.splitlines()
    non_empty = [l for l in lines if l.strip()]
    if non_empty:
        # Check if all non-empty lines share common leading indentation
        min_indent = min(len(l) - len(l.lstrip()) for l in non_empty)
        if min_indent > 0:
            lines = [l[min_indent:] if len(l) >= min_indent else l for l in lines]

    # Fix unexpected leading whitespace before top-level assignments/imports
    fixed_lines = []
    for line in lines:
        stripped = line.strip()
        if (
            stripped.startswith("import ")
            or stripped.startswith("file_path =")
            or stripped.startswith("file_path_")
            or stripped.startswith("df = pd.read_csv")
            or stripped.startswith("df =")
        ):
            fixed_lines.append(stripped)
        else:
            fixed_lines.append(line)
    code_str = "\n".join(fixed_lines)

    # Fix bug 1: `if 'X' in df[col].str.contains(...)` -> replace with `if (df[col].str.contains(...)).any():`
    code_str = re.sub(
        r"if\s+['\"].*?['\"]\s+in\s+([^\n:]+?\.str\.contains\([^:\n]+\)):[ \t]*",
        r"if (\1).any():",
        code_str
    )
    # Fix bug 2: `df[df[col] == 'keyword']` -> replace with `df[df[col].astype(str).str.contains('keyword', case=False, na=False, regex=False)]`
    code_str = re.sub(
        r"df\[\s*df\[(['\"].*?['\"])\s*\]\s*==\s*(['\"].*?['\"])\s*\]",
        r"df[df[\1].astype(str).str.contains(\2, case=False, na=False, regex=False)]",
        code_str
    )
    # Fix bug 3: Ensure automatic insertion of `.astype(str)` before any `.str.` operations if missing
    code_str = re.sub(
        r"(df\[\s*['\"][^'\"]+['\"]\s*\])(?!\.astype\(str\))\.str\.",
        r"\1.astype(str).str.",
        code_str
    )
    return code_str


def clean_val(val):
    if pd.isna(val):
        raise ValueError("Metric not found in table")
    val_str = str(val).strip()
    if val_str in ['-', '—']:
        return 0.0
    if not val_str or val_str in ['', 'nan', 'NaN', 'None', 'null', 'n/a']:
        raise ValueError("Metric not found in table")
    if isinstance(val, (int, float)): return float(val)
    neg = False
    if val_str.startswith('(') and val_str.endswith(')'):
        neg = True
        val_str = val_str[1:-1].strip()
    if val_str.endswith('%'):
        val_str = val_str[:-1].strip()
    
    # Xử lý dấu phẩy thập phân kiểu Việt Nam (ví dụ '27,78' hoặc '35,0')
    if ',' in val_str and '.' not in val_str:
        parts = val_str.split(',')
        if len(parts) == 2 and len(parts[1]) in (1, 2):
            val_str = val_str.replace(',', '.')
        else:
            val_str = val_str.replace(',', '')
    else:
        val_str = val_str.replace(',', '')

    if '.' in val_str:
        parts = val_str.split('.')
        if len(parts) > 2 or (len(parts) == 2 and len(parts[1]) == 3):
            val_str = val_str.replace('.', '')
    try:
        res = float(val_str)
        return -res if neg else res
    except Exception:
        raise ValueError("Metric not found in table")


def extract_value(row, preferred_col, _df=None, _row_idx=None, abs_val=False):
    """Extract a numeric value from a row, trying preferred_col first then fallback columns.
    
    Enhanced: Supports integer column indices as preferred_col and optional abs_val conversion.
    """
    if isinstance(preferred_col, (int, np.integer)):
        if hasattr(row, 'index') and 0 <= preferred_col < len(row.index):
            preferred_col = row.index[preferred_col]
        elif isinstance(row, pd.DataFrame) and not row.empty and 0 <= preferred_col < len(row.columns):
            preferred_col = row.columns[preferred_col]

    _meta = {'Ma_Doanh_Nghiep', 'Ten_Doanh_Nghiep', 'Nam_Tai_Chinh', 'Loai_Bao_Cao', 'Ten_Bang', 'Don_Vi_Tinh', 'Tep_Nguon', 'Cột_0', '0', 'STT'}
    avail_cols = []
    if hasattr(row, 'index'):
        avail_cols = [c for c in row.index if c not in _meta]
    elif isinstance(row, pd.DataFrame) and not row.empty:
        avail_cols = [c for c in row.columns if c not in _meta]

    ordered_fallbacks = ['Năm nay', '2025', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '31/12/2024', '31/12/2023', '1', '2', '3', '4', '5']
    seen = set()
    cols_to_try = []
    for c in [preferred_col] + ordered_fallbacks + avail_cols:
        if c and c not in seen:
            seen.add(c)
            cols_to_try.append(c)
    
    # Try extracting from the current row first
    for c in cols_to_try:
        if hasattr(row, 'index') and c in row.index:
            try:
                v = clean_val(row[c])
                return abs(v) if abs_val else v
            except ValueError:
                continue
        elif isinstance(row, pd.DataFrame) and c in row.columns and not row.empty:
            try:
                v = clean_val(row[c].iloc[0])
                return abs(v) if abs_val else v
            except ValueError:
                continue
    
    target_df = _df
    target_idx = _row_idx
    if target_idx is None and hasattr(row, 'name') and isinstance(row.name, (int, np.integer)):
        target_idx = int(row.name)

    # Special handling for VAMC special bonds table (Q50)
    if target_df is not None:
        try:
            df_str = target_df.astype(str).to_string().lower()
            if "vamc" in df_str or "trái phiếu đặc biệt" in df_str:
                for idx in range(len(target_df) - 1, -1, -1):
                    r_cand = target_df.iloc[idx]
                    for c in cols_to_try:
                        if c in r_cand.index:
                            try:
                                v_cand = clean_val(r_cand[c])
                                if v_cand != 0.0:
                                    return abs(v_cand) if abs_val else v_cand
                            except ValueError:
                                pass
        except Exception:
            pass

    # Fallback: If row has all NaN values (hierarchical parent row),
    # try the next 2 rows (child rows) which may contain the actual data (Q12).
    if target_df is not None and target_idx is not None:
        for offset in [1, 2]:
            next_idx = target_idx + offset
            if next_idx < len(target_df):
                next_row = target_df.iloc[next_idx]
                for c in cols_to_try:
                    if hasattr(next_row, 'index') and c in next_row.index:
                        try:
                            v_next = clean_val(next_row[c])
                            return abs(v_next) if abs_val else v_next
                        except ValueError:
                            continue
    
    raise ValueError("Metric not found in table")


def aggregate_top_k_results(results: List[Dict[str, Any]], is_percentage: bool = False) -> Dict[str, Any]:
    """Lọc các kết quả trích xuất hợp lệ từ Top 5 bảng và lấy giá trị lớn nhất."""
    valid_candidates = []
    for r in results:
        val = r.get("value")
        if val is not None and isinstance(val, (int, float, np.integer, np.floating)):
            if not np.isnan(val):
                valid_candidates.append((float(val), r.get("table_name", ""), r.get("csv_path", "")))
        elif isinstance(val, dict) and "data" in val:
            d = val["data"]
            if isinstance(d, (int, float, np.integer, np.floating)) and not np.isnan(d):
                valid_candidates.append((float(d), r.get("table_name", ""), r.get("csv_path", "")))

    if not valid_candidates:
        raise ValueError("Metric not found across Top 5 candidate tables")

    # If percentage query and there are valid percentage candidates (0 <= val <= 100), filter for them
    if is_percentage:
        pct_candidates = [c for c in valid_candidates if 0.0 <= c[0] <= 100.0]
        if pct_candidates:
            valid_candidates = pct_candidates

    # Chọn giá trị lớn nhất theo độ lớn (magnitude / absolute value)
    best_tuple = max(valid_candidates, key=lambda x: abs(x[0]))
    max_val, best_name, best_path = best_tuple

    return {
        "type": "scalar",
        "data": max_val,
        "source_table": best_name,
        "source_path": best_path,
        "candidate_count": len(valid_candidates),
    }


def execute_code_on_table(code_str: str, file_path: str, all_tables: Optional[List[Dict]] = None) -> Any:
    """Execute python snippet safely on a specific table file."""
    all_tables = all_tables or []
    df_loaded = None
    if file_path:
        df_loaded = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)

    from pathlib import Path
    import pathlib
    exec_globals = {
        "pd": pd,
        "np": np,
        "pandas": pd,
        "numpy": np,
        "Path": Path,
        "pathlib": pathlib,
        "file_path": file_path,
        "df": df_loaded,
        "clean_val": clean_val,
        "extract_value": extract_value,
    }
    for tbl in all_tables:
        csv_p = tbl.get("csv_path", "")
        nam = tbl.get("Nam_Tai_Chinh", "")
        if csv_p and nam:
            exec_globals[f"file_path_{nam}"] = csv_p

    exec(code_str, exec_globals)
    result_val = exec_globals.get("result")
    if result_val is None:
        raise ValueError("Biến `result` không được tìm thấy sau khi thực thi mã.")
    return result_val


def executor_node(state: AgentState, cfg: Optional[Config] = None) -> AgentState:
    """LangGraph Node 5: Safely execute generated Pandas code and capture result.
    
    Supports Multi-Table Top-5 Candidate Execution & Max Value Aggregation.
    """
    cfg = cfg or default_config
    start_time = time.time()

    code_str = state.get("generated_code", "").strip()
    code_str = sanitize_code_str(code_str)

    discovered_tables = state.get("discovered_tables", [])
    top_candidates = state.get("top_k_candidates", discovered_tables[:5])
    retry_count = state.get("retry_count", 0)

    if not code_str:
        return {
            **state,
            "status": "error",
            "error_traceback": "No code generated to execute.",
            "retry_count": retry_count + 1,
        }

    try:
        # Step 1: Validate AST
        validate_ast(code_str)

        print(f"⚙️ [Executor] Đang thực thi mã Pandas trên Top {len(top_candidates)} bảng ứng viên...")

        multi_table_results = []
        last_error_tb = None

        # Execute across top candidate tables
        for idx, tbl in enumerate(top_candidates):
            c_path = tbl.get("csv_path", "")
            t_name = tbl.get("Ten_Bang", Path(c_path).stem if c_path else f"Table_{idx+1}")
            try:
                res_val = execute_code_on_table(code_str, c_path, all_tables=discovered_tables)
                multi_table_results.append({
                    "table_idx": idx,
                    "table_name": t_name,
                    "csv_path": c_path,
                    "value": res_val,
                    "status": "success",
                })
                print(f"   ✅ Bảng #{idx+1} ({t_name}): Trích xuất thành công -> {res_val}")
            except Exception as e:
                last_error_tb = traceback.format_exc()
                multi_table_results.append({
                    "table_idx": idx,
                    "table_name": t_name,
                    "csv_path": c_path,
                    "value": None,
                    "status": "error",
                    "error": str(e),
                })
                print(f"   ⚠️ Bảng #{idx+1} ({t_name}): {e}")

        # Step 2: Aggregate results and select Max Value
        successful_candidates = [r for r in multi_table_results if r["status"] == "success"]

        user_query = state.get("user_query", "")
        is_pct = any(k in str(user_query).lower() for k in ["tỷ lệ", "phần trăm", "%", "biểu quyết", "lợi ích", "sở hữu"])

        if successful_candidates:
            aggregated = aggregate_top_k_results(multi_table_results, is_percentage=is_pct)
            formatted = format_result(aggregated["data"])
            
            print(f"\n🏆 [Executor] Top-5 Max Aggregator THÀNH CÔNG!")
            print(f"   • Giá trị lớn nhất: {aggregated['data']} (Từ bảng: {aggregated['source_table']})")
            print(f"   • Số bảng ứng viên hợp lệ: {aggregated['candidate_count']}/{len(top_candidates)}")

            latency = time.time() - start_time
            node_latencies = state.get("node_latencies", {})
            node_latencies["executor"] = round(latency, 3)

            return {
                **state,
                "execution_result": formatted,
                "aggregated_value": aggregated["data"],
                "multi_table_results": multi_table_results,
                "error_traceback": None,
                "status": "success",
                "node_latencies": node_latencies,
            }
        else:
            raise ValueError(f"Không tìm thấy chỉ tiêu trong toàn bộ Top {len(top_candidates)} bảng ứng viên.")

    except Exception as e:
        latency = time.time() - start_time
        node_latencies = state.get("node_latencies", {})
        node_latencies["executor"] = round(latency, 3)

        tb_str = traceback.format_exc()

        return {
            **state,
            "status": "error",
            "error_traceback": tb_str,
            "multi_table_results": multi_table_results if 'multi_table_results' in locals() else [],
            "retry_count": retry_count + 1,
            "node_latencies": node_latencies,
        }


## 🌐 Section 5: LangGraph Workflow & Edge Routing
Xây dựng đồ thị trạng thái Agent StateGraph và thiết lập điều kiện Reflection Loop.

In [30]:
"""LangGraph StateGraph Definition for Cocopila Pandas Data Agent Pipeline."""

from typing import Literal, Optional
from langgraph.graph import StateGraph, END

from pipeline.src.state import AgentState
from pipeline.src.config import Config, config as default_config
from pipeline.src.nodes.query_parser import parse_query_node
from pipeline.src.nodes.data_discovery import data_discovery_node
from pipeline.src.nodes.schema_mapper import schema_mapper_node
from pipeline.src.nodes.code_generator import code_generator_node
from pipeline.src.nodes.executor import executor_node

def route_after_discovery(state: AgentState) -> Literal["schema_mapper", "__end__"]:
    """Route workflow after data discovery node."""
    if state.get("status") == "error":
        return END
    return "schema_mapper"


def route_after_schema_mapper(state: AgentState) -> Literal["code_generator", "__end__"]:
    """Route workflow after schema mapper node."""
    if state.get("status") == "error":
        return END
    return "code_generator"


def route_after_execution(state: AgentState, cfg: Optional[Config] = None) -> Literal["code_generator", "__end__"]:
    """Conditional edge for Reflection Debugging Loop."""
    cfg = cfg or default_config
    status = state.get("status")
    retry_count = state.get("retry_count", 0)

    if status == "success":
        return END

    if status == "error" and retry_count < cfg.MAX_RETRIES:
        print(f"🔄 Reflection Loop Activated! Retrying code generation ({retry_count}/{cfg.MAX_RETRIES})...")
        return "code_generator"

    return END


def create_cocopila_graph(cfg: Optional[Config] = None):
    """Construct and compile the LangGraph workflow graph."""
    cfg = cfg or default_config

    workflow = StateGraph(AgentState)

    # Add Nodes
    workflow.add_node("query_parser", lambda state: parse_query_node(state, cfg))
    workflow.add_node("data_discovery", lambda state: data_discovery_node(state, cfg))
    workflow.add_node("schema_mapper", lambda state: schema_mapper_node(state, cfg))
    workflow.add_node("code_generator", lambda state: code_generator_node(state, cfg))
    workflow.add_node("executor", lambda state: executor_node(state, cfg))

    # Define Workflow Edges
    workflow.set_entry_point("query_parser")
    workflow.add_edge("query_parser", "data_discovery")

    workflow.add_conditional_edges(
        "data_discovery",
        route_after_discovery,
        {
            "schema_mapper": "schema_mapper",
            END: END,
        }
    )

    workflow.add_conditional_edges(
        "schema_mapper",
        route_after_schema_mapper,
        {
            "code_generator": "code_generator",
            END: END,
        }
    )

    workflow.add_edge("code_generator", "executor")

    workflow.add_conditional_edges(
        "executor",
        lambda state: route_after_execution(state, cfg),
        {
            "code_generator": "code_generator",
            END: END,
        }
    )

    return workflow.compile()


print('✅ Section 5: LangGraph Workflow loaded!')


✅ Section 5: LangGraph Workflow loaded!


## 🧪 Section 6: Dataset Linking & Running Agent Test

In [31]:
# 2. Dò tìm và liên kết trực tiếp Qdrant Local DB từ đường dẫn chỉ định trên Kaggle
import os
import shutil
from pathlib import Path

# Các đường dẫn khả thi trên Kaggle (bao gồm cả URL trình duyệt và đường dẫn hệ thống thực tế)
possible_dataset_dirs = [
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output/r2AI_data"),
    Path("/kaggle/input/r2-ai-output"),
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output"),
]

if Path("/kaggle/input").exists():
    print("🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...")
    dataset_dir = None
    
    # 1. Thử các đường dẫn chỉ định trước
    for d in possible_dataset_dirs:
        if (d / "qdrant_local_db").exists():
            dataset_dir = d
            print(f"📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: {d / 'qdrant_local_db'}")
            break
            
    # 2. Nếu không thấy, dùng quét nông (shallow search) để tìm kiếm tự động
    if not dataset_dir:
        qdrant_found = []
        for depth_pattern in [
            "*/qdrant_local_db", 
            "*/*/qdrant_local_db", 
            "*/*/*/qdrant_local_db", 
            "*/*/*/*/qdrant_local_db", 
            "*/*/*/*/*/qdrant_local_db"
        ]:
            qdrant_found.extend(list(Path("/kaggle/input").glob(depth_pattern)))
            if qdrant_found:
                dataset_dir = qdrant_found[0].parent
                print(f"📦 Đã tìm thấy Qdrant DB bằng wildcard tại: {qdrant_found[0]}")
                break
            
    if dataset_dir:
        rag_module_dir = Path("rag_module").resolve()
        rag_module_dir.mkdir(exist_ok=True)
        
        # Symlink cho các file chỉ đọc (ViFinQA, bm25, code_stock)
        for item in ["bm25_index.pkl", "code_stock.csv", "ViFinQA"]:
            src = dataset_dir / item
            dst = rag_module_dir / item
            
            # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError
            if dst.exists() or dst.is_symlink():
                try:
                    if dst.is_symlink() or dst.is_file():
                        dst.unlink()
                    else:
                        shutil.rmtree(dst)
                except Exception as e:
                    print(f"   ⚠️ Cannot remove old {item}: {e}")
                    
            if src.exists() and not dst.exists():
                try:
                    os.symlink(src, dst)
                    print(f"   🔗 Created symlink: {dst} -> {src}")
                except Exception as e:
                    print(f"   ⚠️ Cannot symlink {item}: {e}")
        
        # Copy vật lý cho qdrant_local_db vì Qdrant yêu cầu quyền ghi (.lock file)
        qdrant_src = dataset_dir / "qdrant_local_db"
        qdrant_dst = rag_module_dir / "qdrant_local_db"
        
        # Xóa liên kết cũ/hỏng nếu có để tránh lỗi FileExistsError khi copytree
        if qdrant_dst.exists() or qdrant_dst.is_symlink():
            try:
                if qdrant_dst.is_symlink() or qdrant_dst.is_file():
                    qdrant_dst.unlink()
                else:
                    shutil.rmtree(qdrant_dst)
            except Exception as e:
                print(f"   ⚠️ Cannot remove old qdrant_local_db: {e}")
                
        if qdrant_src.exists() and not qdrant_dst.exists():
            print("   ⏳ Đang copy Qdrant DB sang thư mục làm việc để cấp quyền ghi (chỉ mất vài chục giây cho lần đầu)...")
            shutil.copytree(qdrant_src, qdrant_dst)
            print(f"   ✅ Đã copy xong Qdrant DB tới: {qdrant_dst}")
    else:
        print("❌ Không tìm thấy thư mục dataset trên Kaggle! Hãy kiểm tra xem bạn đã đính kèm dataset vào Notebook chưa.")
else:
    print("💻 Chạy local, sử dụng dữ liệu có sẵn tại rag_module/")

🔍 Đang kiểm tra Kaggle Input Dataset cho Qdrant DB...
📦 Đã tìm thấy Qdrant DB tại đường dẫn chỉ định: /kaggle/input/datasets/duymcminh/r2-ai-output/qdrant_local_db
   🔗 Created symlink: /kaggle/working/r2AI_2026/rag_module/bm25_index.pkl -> /kaggle/input/datasets/duymcminh/r2-ai-output/bm25_index.pkl
   🔗 Created symlink: /kaggle/working/r2AI_2026/rag_module/ViFinQA -> /kaggle/input/datasets/duymcminh/r2-ai-output/ViFinQA
   ⏳ Đang copy Qdrant DB sang thư mục làm việc để cấp quyền ghi (chỉ mất vài chục giây cho lần đầu)...
   ✅ Đã copy xong Qdrant DB tới: /kaggle/working/r2AI_2026/rag_module/qdrant_local_db


In [32]:
# 3. Khởi tạo Agent và thực thi danh sách câu hỏi kiểm thử (LANGGRAPH STATEGRAPH WORKFLOW)
import time
import random
import json
from pathlib import Path

# Khởi tạo LangGraph workflow đã được định nghĩa tại Cell 25
app = create_cocopila_graph(config)

# Nạp danh sách câu hỏi kiểm thử từ dataset
repo_dir = Path("/kaggle/working/r2AI_2026")
if not repo_dir.exists():
    repo_dir = Path.cwd()

possible_qa_paths = [
    repo_dir / "rag_module" / "ViFinQA" / "questions" / "questions.jsonl",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    Path("/kaggle/input/r2-ai-output/rag_module/ViFinQA/questions/questions.jsonl"),
    repo_dir / "rag_module" / "ViFinQA" / "ViFinQA_QA.json",
    Path("/kaggle/input/datasets/duymcminh/r2-ai-output/r2AI_data/ViFinQA/ViFinQA_QA.json"),
    Path("rag_module/ViFinQA/ViFinQA_QA.json"),
]

qa_file = None
for p in possible_qa_paths:
    if p.exists():
        qa_file = p
        break

qa_data = []
if qa_file and qa_file.exists():
    with open(qa_file, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if qa_file.suffix == ".jsonl" or "\n" in content:
            for line in content.splitlines():
                if line.strip():
                    try:
                        qa_data.append(json.loads(line))
                    except Exception:
                        pass
        else:
            try:
                qa_data = json.loads(content)
            except Exception:
                pass
    print(f"📋 Đã tải thành công {len(qa_data)} câu hỏi từ dataset!")
else:
    print("⚠️ Không tìm thấy file ViFinQA_QA.json, sử dụng câu hỏi mẫu mặc định.")
    qa_data = [
        {"id": 2, "question": "Số dư cho vay khách hàng ngành Thương mại của công ty mẹ Ngân hàng TMCP Á Châu (ACB) cuối năm 2022 là bao nhiêu triệu đồng?"},
        {"id": 12, "question": "Vốn cổ phần đã phát hành của Tập đoàn Dệt may Việt Nam (VGT) năm 2024 là bao nhiêu triệu đồng?"},
        {"id": 16, "question": "Vay và nợ thuê tài chính ngắn hạn của Tập đoàn CEO năm 2025 là bao nhiêu triệu đồng?"},
    ]

random.seed(67)
sample_questions = qa_data[:20] if len(qa_data) >= 20 else qa_data

results_summary = []

for idx, q_item in enumerate(sample_questions, 1):
    q_id = q_item.get("id", idx)
    q_text = q_item.get("question", "")
    print(f"\n[{idx}/{len(sample_questions)}] ❓ Câu hỏi ID {q_id}: {q_text}")
    print("-" * 60)
    
    try:
        final_state = app.invoke({"user_query": q_text})
        exec_res = final_state.get("execution_result")
        status = final_state.get("status")
        
        if status == "success":
            val_out = final_state.get("aggregated_value")
            if val_out is None and isinstance(exec_res, dict):
                val_out = exec_res.get("data")
            print(f"✅ ID {q_id}: THÀNH CÔNG -> Kết quả: {val_out}")
            results_summary.append({
                "id": q_id,
                "question": q_text,
                "status": "success",
                "result": exec_res,
                "aggregated_value": val_out,
                "generated_code": final_state.get("generated_code", ""),
                "parsed_query": final_state.get("parsed_query"),
                "discovered_tables": final_state.get("discovered_tables", []),
            })
        else:
            err_msg = final_state.get("error_traceback") or final_state.get("error_message") or "Unknown error"
            print(f"❌ ID {q_id}: THẤT BẠI -> {err_msg.splitlines()[-1] if err_msg else err_msg}")
            results_summary.append({
                "id": q_id,
                "question": q_text,
                "status": "error",
                "error": err_msg,
                "generated_code": final_state.get("generated_code", ""),
                "parsed_query": final_state.get("parsed_query"),
                "discovered_tables": final_state.get("discovered_tables", []),
            })
    except Exception as e:
        print(f"   💥 Lỗi thực thi LangGraph cho câu hỏi ID {q_id}: {e}")
        results_summary.append({"id": q_id, "question": q_text, "status": "error", "error": str(e)})

print("\n" + "=" * 80)
print("📊 TỔNG HỢP KẾT QUẢ KIỂM THỬ PIPELINE (LANGGRAPH):")
print("=" * 80)
success_count = sum(1 for r in results_summary if r["status"] == "success")
print(f"Tổng số câu hỏi: {len(results_summary)} | Thành công: {success_count} | Thất bại: {len(results_summary) - success_count}")
for r in results_summary:
    status_emoji = "✅" if r["status"] == "success" else "❌"
    val_str = f" -> {r.get('aggregated_value') or r.get('result')}" if r["status"] == "success" else ""
    print(f"{status_emoji} ID {r['id']}: {r['status'].upper()}{val_str}")


📋 Đã tải thành công 1012 câu hỏi từ dataset!

[1/20] ❓ Câu hỏi ID 1: Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng không Vietjet (VJC) là bao nhiêu triệu đồng?
------------------------------------------------------------

🔍 [Query Parser] Đang phân tích câu hỏi: 'Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng không Vietjet (VJC) là bao nhiêu triệu đồng?'
📊 [Kết quả - Query Parser]:
   Công ty: VJC
   Năm: ['2018']
   Nội dung: Lãi tiền gửi
   Thao tác: trich_xuat
   Tiêu chí phụ: None
   Nhân sự: None


🔍 [Data Discovery] Bắt đầu tìm kiếm dữ liệu...
   - Công ty: 'VJC'
   - Số năm: ['2018']
   - Nội dung cần tìm (đã làm sạch): 'Lãi tiền gửi' (gốc: 'Lãi tiền gửi')
   - Thao tác: trich_xuat
   - Tra cứu bảng cho năm 2018...
   📋 Danh sách 5 bảng ứng viên Top-K từ Search Engine (Năm 2018):
      #1 RRF: 0.330550 | DenseRank: 7 | SparseRank: 4 ✅ [Matched Cột 'Cột_0']
         File: VJC_financial_statements_2018_separate_table_7@line_229.csv
         Tên bảng: Các thuyết minh đính kèm là b

KeyboardInterrupt: 

## 💾 Section 7: Lưu trữ & Kiểm thử thủ công kết quả Code Generator (Export, Zip & Manual Verification)
Mục này cung cấp các công cụ hỗ trợ kiểm thử thủ công và debug:
1. **Lưu trữ kết quả**: Xuất toàn bộ kết quả chạy kèm mã nguồn Python (`generated_code`) ra file JSON và CSV.
2. **Xuất Scripts riêng biệt**: Tự động sinh file script `.py` cho từng câu hỏi vào thư mục `manual_test_scripts/` để kiểm tra độc lập.
3. **Nén ZIP tự động**: Đóng gói toàn bộ output (`.json`, `.csv`, `.py` scripts) thành một file `.zip` duy nhất (`codegen_manual_tests.zip`) để dễ dàng tải về từ Kaggle/Local.
4. **Bảng tổng hợp trực quan**: DataFrame hiển thị trạng thái, kết quả và độ dài mã của từng câu hỏi.
5. **Hàm kiểm thử thủ công `inspect_and_run(q_id)`**: Cho phép in toàn bộ mã nguồn sinh ra, xem các bảng dữ liệu liên quan và re-run trực tiếp mã nguồn trong Sandbox AST.

In [33]:
# 1. Khởi tạo thư mục lưu trữ kết quả kiểm thử
import json
import shutil
import zipfile
import pandas as pd
from pathlib import Path

output_dir = Path("/kaggle/working/manual_test_scripts") if Path("/kaggle/working").exists() else Path("./manual_test_scripts")
output_dir.mkdir(parents=True, exist_ok=True)

results_file_json = output_dir / "codegen_results.json"
results_file_csv = output_dir / "codegen_results.csv"

# 2. Lưu toàn bộ kết quả vào file JSON (đầy đủ metadata, code, tracebacks)
with open(results_file_json, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

# 3. Xuất từng đoạn code sinh ra thành các file script độc lập kèm header metadata (tiền tố script_q_ để tránh xung đột với pytest)
for item in results_summary:
    q_id = item.get("id")
    status = item.get("status", "unknown")
    code = item.get("generated_code", "")
    question = item.get("question", "")
    result = item.get("result")
    err = item.get("error")
    attempts = item.get("attempts", 1)
    
    script_path = output_dir / f"script_q_{q_id}_{status}.py"
    header_comment = f'''"""
======================================================================
QUESTION ID : {q_id}
STATUS      : {status} (Attempts: {attempts})
QUESTION    : {question}
RESULT      : {json.dumps(result, ensure_ascii=False) if result else "None"}
ERROR       : {err if err else "None"}
======================================================================
"""
'''
    with open(script_path, "w", encoding="utf-8") as sf:
        sf.write(header_comment + "\n" + (code if code else "# Không có mã nguồn Python nào được sinh ra."))

# 4. Sao chép file log .txt vào thư mục kết quả để đóng gói
log_file_dest = output_dir / "pipeline_execution.txt"
if LOG_FILE_PATH.exists():
    shutil.copy(LOG_FILE_PATH, log_file_dest)

# 5. Tạo bảng DataFrame tổng hợp kết quả
summary_rows = []
for item in results_summary:
    res = item.get("result")
    res_val = res.get("data") if isinstance(res, dict) else res
    res_type = res.get("type") if isinstance(res, dict) else type(res).__name__
    code_text = item.get("generated_code", "")
    
    summary_rows.append({
        "ID": item.get("id"),
        "Câu hỏi": item.get("question", "")[:60] + ("..." if len(item.get("question", "")) > 60 else ""),
        "Trạng thái": "✅ SUCCESS" if item.get("status") == "success" else "❌ ERROR",
        "Số lần thử": item.get("attempts", 1),
        "Kết quả": str(res_val)[:40] if res_val is not None else "None",
        "Kiểu dữ liệu": res_type,
        "Lỗi": item.get("error") or "Không có",
        "Độ dài mã (dòng)": len(code_text.splitlines()) if code_text else 0,
    })

df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(results_file_csv, index=False, encoding="utf-8-sig")

# 6. Nén toàn bộ thư mục output (bao gồm file log .txt) thành 1 file ZIP duy nhất
working_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
zip_file_path = (working_dir / "codegen_manual_tests.zip").resolve()

with zipfile.ZipFile(zip_file_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file in sorted(output_dir.rglob("*")):
        if file.is_file():
            zf.write(file, arcname=file.name)

zip_size_kb = round(zip_file_path.stat().st_size / 1024, 2) if zip_file_path.exists() else 0
log_size_kb = round(LOG_FILE_PATH.stat().st_size / 1024, 2) if LOG_FILE_PATH.exists() else 0

print("=" * 80)
print("💾 ĐÃ LƯU & NÉN TOÀN BỘ KẾT QUẢ KIỂM THỬ THÀNH CÔNG!")
print("=" * 80)
print(f"📁 Thư mục mã nguồn kiểm thử : {output_dir.resolve()}")
print(f"📝 File log toàn bộ quá trình: {LOG_FILE_PATH.resolve()} ({log_size_kb} KB)")
print(f"📄 File kết quả JSON         : {results_file_json.resolve()}")
print(f"📄 File kết quả CSV          : {results_file_csv.resolve()}")
print(f"📦 FILE ZIP TỔNG HỢP OUTPUT  : {zip_file_path} ({zip_size_kb} KB)")
print("=" * 80)

# Hiển thị bảng tổng hợp
display(df_summary)

# 7. Hàm tiện ích để kiểm thử thủ công & Re-run từng câu hỏi
def inspect_and_run(target_q_id: int):
    """Xem chi tiết câu hỏi, mã nguồn Python sinh ra và thực thi lại trong Sandbox AST."""
    matched = [r for r in results_summary if r.get("id") == target_q_id]
    if not matched:
        print(f"❌ Không tìm thấy câu hỏi với ID = {target_q_id} trong kết quả kiểm thử.")
        return
    
    item = matched[0]
    print("\n" + "=" * 80)
    print(f"🔍 CHI TIẾT KIỂM THỬ THỦ CÔNG - CÂU HỎI ID: {target_q_id}")
    print("=" * 80)
    print(f"❓ Câu hỏi    : {item.get('question')}")
    print(f"📊 Trạng thái : {item.get('status')}")
    print(f"🔄 Số lần thử : {item.get('attempts', 1)}")
    print(f"🎯 Kết quả cũ : {item.get('result')}")
    if item.get("error"):
        print(f"⚠️ Thông báo lỗi: {item.get('error')}")
    
    tables = item.get("discovered_tables", [])
    if tables:
        print(f"\n📂 Các bảng dữ liệu liên quan ({len(tables)} bảng):")
        for t in tables:
            print(f"   - Năm {t.get('Nam_Tai_Chinh', 'N/A')}: {t.get('csv_path')}")
            
    code = item.get("generated_code", "")
    print("\n💻 MÃ NGUỒN PYTHON ĐƯỢC SINH RA:")
    print("-" * 80)
    if code:
        print(code)
    else:
        print("# (Không có mã nguồn nào được sinh ra)")
        print("-" * 80)
        return
    print("-" * 80)
    
    print("\n⚡ TIẾN HÀNH THỰC THI LẠI TRONG SANDBOX (RE-RUN):")
    re_state = {
        "user_query": item.get("question", ""),
        "generated_code": code,
        "discovered_tables": tables,
        "status": "pending",
        "error_traceback": None,
        "execution_result": None
    }
    try:
        re_output = executor_node(re_state, config)
        if re_output.get("status") == "success":
            print(f"👉 Re-run THÀNH CÔNG! Kết quả: {re_output.get('execution_result')}")
        else:
            print(f"👉 Re-run THẤT BẠI!")
            print(f"⚠️ Traceback: {re_output.get('error_traceback')}")
    except Exception as re_err:
        print(f"💥 Ngoại lệ khi re-run: {re_err}")

print("\n💡 HƯỚNG DẪN KIỂM THỬ THỦ CÔNG:")
print("   Để xem mã nguồn và chạy thử lại bất kỳ câu hỏi nào, gọi hàm:")
first_id = results_summary[0]['id'] if results_summary else 1
print(f"   >>> inspect_and_run({first_id})")


💾 ĐÃ LƯU & NÉN TOÀN BỘ KẾT QUẢ KIỂM THỬ THÀNH CÔNG!
📁 Thư mục mã nguồn kiểm thử : /kaggle/working/manual_test_scripts
📝 File log toàn bộ quá trình: /kaggle/working/pipeline_execution.txt (346.39 KB)
📄 File kết quả JSON         : /kaggle/working/manual_test_scripts/codegen_results.json
📄 File kết quả CSV          : /kaggle/working/manual_test_scripts/codegen_results.csv
📦 FILE ZIP TỔNG HỢP OUTPUT  : /kaggle/working/codegen_manual_tests.zip (44.18 KB)


,ID,Câu hỏi,Trạng thái,Số lần thử,Kết quả,Kiểu dữ liệu,Lỗi,Độ dài mã (dòng)
0,1,Lãi tiền gửi năm 2018 của công ty mẹ CTCP Hàng...,✅ SUCCESS,1,-208253201298.0,scalar,Không có,36
1,2,Số dư cho vay khách hàng ngành Thương mại của ...,✅ SUCCESS,1,410003122.0,scalar,Không có,36
2,3,Chi phí dự phòng của Ngân hàng TMCP Sài Gòn Tà...,✅ SUCCESS,1,5947205.0,scalar,Không có,36
3,4,Lợi nhuận sau thuế của CTCP Chứng khoán FPT nă...,✅ SUCCESS,1,3461017594892.0,scalar,Không có,36
4,5,Chi phí phạt của công ty mẹ SCR năm 2017 là ba...,✅ SUCCESS,1,11242504352.0,scalar,Không có,36
5,6,Lưu chuyển tiền thuần từ hoạt động kinh doanh ...,✅ SUCCESS,1,427739490909.0,scalar,Không có,36
6,7,"Quỹ khen thưởng, phúc lợi của HT1 cuối năm 201...",✅ SUCCESS,1,57764463052.0,scalar,Không có,36
7,8,Chi phí lương và các khoản khác theo lương của...,✅ SUCCESS,1,30686828047.0,scalar,Không có,36



💡 HƯỚNG DẪN KIỂM THỬ THỦ CÔNG:
   Để xem mã nguồn và chạy thử lại bất kỳ câu hỏi nào, gọi hàm:
   >>> inspect_and_run(1)


## 📦 Section 8: Tổng hợp & Xuất file Submission.json (Định dạng chuẩn Ban Tổ chức)

In [34]:
# ==============================================================================
# SECTION 8: TỔNG HỢP & XUẤT FILE SUBMISSION.JSON (ĐỊNH DẠNG CHUẨN BAN TỔ CHỨC)
# ==============================================================================
import os
import re
import json
import pandas as pd
from pathlib import Path
from typing import Dict, List, Any, Optional

# 1. Thư mục làm việc & Đường dẫn file submission.json
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUBMISSION_JSON_PATH = (WORKING_DIR / "submission.json").resolve()

# 2. Nạp dữ liệu kết quả từ results_summary (hoặc fallback từ codegen_results.json)
items_to_process = []
if "results_summary" in globals() and isinstance(results_summary, list) and len(results_summary) > 0:
    items_to_process = results_summary
    print(f"📋 Sử dụng dữ liệu từ biến results_summary trong bộ nhớ ({len(items_to_process)} câu hỏi).")
else:
    candidate_json_paths = [
        WORKING_DIR / "manual_test_scripts" / "codegen_results.json",
        Path("./manual_test_scripts/codegen_results.json"),
        Path("r2AI_2026/notebooks/codegen_results.json"),
        Path("codegen_results.json"),
    ]
    for p in candidate_json_paths:
        if p.exists():
            with open(p, "r", encoding="utf-8") as f:
                items_to_process = json.load(f)
            print(f"📋 Đã nạp thành công {len(items_to_process)} câu hỏi từ file {p.resolve()}.")
            break

if not items_to_process:
    print("⚠️ Cảnh báo: Không tìm thấy dữ liệu kết quả nào để xuất submission.json.")

# 3. Các hàm bổ trợ trích xuất & chuẩn hóa
def extract_doc_id(file_name_or_path: str) -> str:
    """Trích xuất ID tài liệu (ví dụ: VJC_2021) từ đường dẫn hoặc tên bảng."""
    if not file_name_or_path:
        return ""
    name = Path(file_name_or_path).name
    name = re.sub(r"#table_\d+", "", name)
    name = re.sub(r"@line_\d+", "", name)
    m = re.search(r"([A-Z0-9]+_\d{4})", name)
    if m:
        return m.group(1)
    return name.split(".")[0].split("_table_")[0]

def get_table_start_line(table_info: Dict[str, Any], table_num: int) -> int:
    """Rút trích trực tiếp số dòng (@line_XXX) từ tên bảng hoặc Tep_Nguon.
    Không quét/đếm dòng qua file OCR text để tối ưu thời gian thực thi.
    """
    if isinstance(table_info, dict):
        for key in ["Tep_Nguon", "Ten_Bang", "table_name", "csv_path"]:
            val = str(table_info.get(key, ""))
            if val:
                m = re.search(r"@?line[_\s:]*(\d+)", val, re.IGNORECASE)
                if m:
                    return int(m.group(1))
    return max(1, (table_num + 1) * 30)

def parse_float_answer(val: Any) -> float:
    """Chuẩn hóa giá trị đáp án về dạng float."""
    if isinstance(val, (int, float)):
        return float(val)
    if isinstance(val, dict):
        val = val.get("data") if val.get("data") is not None else val.get("value")
    if val is None:
        return 0.0

    try:
        s = str(val).strip()
        is_negative = False
        if s.startswith("(") and s.endswith(")"):
            is_negative = True
            s = s[1:-1].strip()

        s = re.sub(r"[^\d.,\-+eE]", "", s)
        if not s or s in ["-", "+", "."]:
            return 0.0

        if "." in s and "," in s:
            if s.rfind(",") > s.rfind("."):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        elif "," in s:
            parts = s.split(",")
            if len(parts) == 2 and len(parts[1]) <= 2:
                s = s.replace(",", ".")
            else:
                s = s.replace(",", "")
        elif s.count(".") > 1:
            s = s.replace(".", "")

        num_val = float(s)
        return -num_val if is_negative else num_val
    except Exception:
        return 0.0

def clean_pandas_query(code_str: str) -> str:
    """Làm sạch chuỗi truy vấn pandas / mã Python."""
    if not code_str:
        return ""
    code = code_str.strip()
    if code.startswith("```"):
        lines = code.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        code = "
".join(lines).strip()
    return code

# 4. Xây dựng danh sách dữ liệu submission chuẩn
submission_data = []

for idx, item in enumerate(items_to_process, 1):
    q_id = int(item.get("id", idx))
    q_text = str(item.get("question", ""))
    
    raw_answer = item.get("aggregated_value")
    if raw_answer is None and item.get("result") is not None:
        raw_answer = item.get("result")
    answer_float = parse_float_answer(raw_answer)

    rel_docs = []
    rel_tables = []
    evidence_list = []
    
    discovered_tables = item.get("discovered_tables", [])
    for t_idx, table_info in enumerate(discovered_tables):
        var_name = f"df{t_idx + 1}"
        csv_path_orig = table_info.get("csv_path", "")
        tep_nguon = table_info.get("Tep_Nguon", "")
        
        csv_filename = Path(csv_path_orig).name if csv_path_orig else ""
        doc_id = extract_doc_id(csv_path_orig or tep_nguon)
        
        t_match = re.search(r"_table_(\d+)", csv_filename) or (re.search(r"#table_(\d+)", tep_nguon) if tep_nguon else None)
        table_num = int(t_match.group(1)) if t_match else t_idx
        
        if not csv_filename:
            csv_filename = f"{doc_id}_table_{table_num}.csv" if doc_id else f"table_{table_num}.csv"

        if doc_id and doc_id not in rel_docs:
            rel_docs.append(doc_id)

        line_number = get_table_start_line(table_info, table_num)
        table_ref = f"{doc_id}|{line_number}" if doc_id else f"table_{table_num}|{line_number}"
        if table_ref not in rel_tables:
            rel_tables.append(table_ref)

        evidence_list.append({
            "variable": var_name,
            "csv_path": f"data/{csv_filename}"
        })

    code_raw = item.get("generated_code", "")
    query_str = clean_pandas_query(code_raw)

    entry = {
        "id": q_id,
        "question": q_text,
        "answer": answer_float,
        "relevant_docs": rel_docs,
        "relevant_tables": rel_tables,
        "evidence": evidence_list,
        "pandas_query": query_str
    }
    submission_data.append(entry)

# 5. Ghi duy nhất file submission.json (không nén ZIP)
with open(SUBMISSION_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(submission_data, f, ensure_ascii=False, indent=2)

print("
" + "=" * 80)
print(f"📄 ĐÃ XUẤT THÀNH CÔNG FILE SUBMISSION.JSON: {SUBMISSION_JSON_PATH}")
print(f"📊 Tổng số bản ghi câu hỏi: {len(submission_data)}")
print("=" * 80)

# 6. Mẫu bản ghi đầu tiên & hiển thị bảng tổng hợp
if submission_data:
    print("
📊 MẪU BẢN GHI SUBMISSION (ITEM 1):")
    print(json.dumps(submission_data[0], indent=2, ensure_ascii=False))

preview_rows = []
for sub in submission_data:
    preview_rows.append({
        "ID": sub["id"],
        "Câu hỏi": sub["question"][:50] + ("..." if len(sub["question"]) > 50 else ""),
        "Answer (float)": sub["answer"],
        "Relevant Docs": ", ".join(sub["relevant_docs"]),
        "Relevant Tables": ", ".join(sub["relevant_tables"]),
        "Evidence CSVs": ", ".join([e["csv_path"] for e in sub["evidence"]]),
        "Query Length": len(sub["pandas_query"])
    })

df_preview = pd.DataFrame(preview_rows)
display(df_preview.head(10))

print("
🎉 HOÀN TẤT XUẤT FILE SUBMISSION.JSON!")


📋 Sử dụng dữ liệu từ biến results_summary trong bộ nhớ (8 câu hỏi).


KeyboardInterrupt: 